# Sentiment-scaled CAPM — Doukas & Han (2021)

A step-by-step replication of *"Sentiment-scaled CAPM and market mispricing"*,
John A. Doukas & Xiao Han, **European Financial Management** 2021, 27:208–243.

This notebook is a teaching companion to `sentiment_capm.py`. Every code cell
below is **the exact code from that module**, in dependency order, with an
explanation of what it does and why. Running the notebook top to bottom gives
you a complete working environment — identical to `%run sentiment_capm.py`,
but with the reasoning visible.

---

## What the paper argues

The static CAPM fails empirically: market betas are not significantly priced,
and the security market line comes out flat. Doukas & Han's explanation is that
beta isn't constant — it moves with **investor sentiment**. Their fix is a
*conditional* CAPM in which the stochastic discount factor's coefficients are
allowed to depend on lagged sentiment.

Starting from a conditional SDF $M_{t+1} = a(z_t) - b(z_t)R^{em}_{t+1}$ with
sentiment $s_t$ as the conditioning instrument $z_t$, and expanding linearly
(Cochrane 2006), they arrive at their **Equation 9** (paper p. 215):

$$E(R_{i,t+1}) = r_f + \beta_{i,s}\lambda_s + \beta_{i,m}\lambda_m + \beta^s_{i,m}\lambda^s_m$$

> "The factors in this model are, the lagged sentiment, the current-period
> market return and lagged sentiment times the current-period market return."

Note carefully: **only sentiment is lagged.** The market excess return is
contemporaneous with the return it prices. That's what makes it a conditional
CAPM rather than a return-forecasting regression.

## The three specifications in this notebook

| Spec | Factors | Driver | Paper |
|---|---|---|---|
| Scaled CAPM (Eq. 9) | $s_{t-1}$, $MktRF_t$, $s_{t-1}\times MktRF_t$ | `run()` | Table 3 |
| Scaled FF3 | $s_{t-1}MktRF_t$, $s_{t-1}SMB_t$, $s_{t-1}HML_t$ | `run_ff3()` | Table 11 |
| Plain FF3 (baseline) | $MktRF_t$, $SMB_t$, $HML_t$ | `run_ff3_plain()` | Table 3, row 2 |

Plus the predictive regression (Table 2), the PLS robustness check (Table 13),
the anomaly portfolios (Tables 8/9), and the state-beta security market line
(Tables 5–7).

## Estimation method: two-pass Fama-MacBeth

**First stage** — one time-series regression per test portfolio $i$, full sample,
giving that portfolio's factor loadings:

$$R_{i,t} - r_{f,t} = a_i + b_{s,i}s_{t-1} + b_{m,i}MktRF_t + b_{sm,i}(s_{t-1}MktRF_t) + e_{i,t}$$

**Second stage** — one cross-sectional regression per month $t$, of that month's
$N$ returns on the $N$ estimated betas, giving a time series of $\lambda_t$:

$$R_{i,t} - r_{f,t} = c_t + \lambda_{s,t}b_{s,i} + \lambda_{m,t}b_{m,i} + \lambda_{sm,t}b_{sm,i} + \alpha_{i,t}$$

The reported risk premia are the time-series averages $\bar\lambda$, with
standard errors from the sample standard deviation of the monthly $\lambda_t$'s.

The paper writes out only the second-stage pricing equation; the first stage is
described in words in its Table 3 note ("The βs are estimated in the first-stage
time-series regressions on different factors"). The first-stage equation above
is the standard FM reconstruction implied by it.

---
## 0. Setup

`DATA_DIR = "00_Data"` is a **relative** path, so the working directory must be
the repository root — the folder that *contains* `00_Data`, not the folder this
notebook lives in. Adjust the path below to match your machine.

In [ ]:
import os

# must be the repo root -- the folder containing 00_Data/
os.chdir("/Users/mandarphatak/Documents/GitHub/Asset_Pricing")
print("working directory:", os.getcwd())
print("00_Data present:", os.path.isdir("00_Data"))

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.decomposition import PCA

DATA_DIR = "00_Data"

---
## 1. Test assets

The cross-sectional tests need a set of portfolios whose average returns the
model is supposed to explain. Three sets are available here, and **the choice
matters enormously** — more than any other decision in this replication.

**25 Size-BM portfolios** — the paper's own test assets, and the field standard.
Formed by independently sorting on market equity and book-to-market into 5×5 bins.

**48 Industry portfolios** — a much harder test. Lewellen, Nagel & Shanken (2010)
recommend adding industries precisely *because* they break models that look fine
on size/BM sorts. The 25 have a strong low-dimensional covariance structure (two
or three principal components explain nearly everything), so almost any three
factors correlated with those components produce a high cross-sectional $R^2$ —
LNS showed this holds even for factors built to be economically meaningless.
Industries have no such shortcut.

**12 and 10 Industry portfolios** — a GICS-style sector cut (`industry12`:
NoDur, Durbl, Manuf, Enrgy, Chems, BusEq, Telcm, Utils, Shops, Hlth, Money,
Other). Note the 10-cut is *not* the 12 with two buckets merged: it drops
Chems and Money as standalone groups and folds BusEq into a broader HiTec, so
they are alternative partitions rather than a nesting.

**32 Size-OP-INV portfolios** — a secondary robustness set.

All are Ken French value-weighted monthly returns in percent.

### Why these rather than a stock screener

Building sector portfolios from a screener introduces **survivorship bias**,
and for sector work it is devastating. A screener returns only *currently
listed* companies, so every firm that went bankrupt, merged or delisted is
silently deleted — and the deletions cluster exactly where the interesting
variation is. A tech sector assembled this way contains only the companies
that survived 2000–02; a financial sector contains no Lehman, no Bear
Stearns, no Washington Mutual. Returns come out biased upward, worst in
precisely the episodes you most want to study.

Ken French's portfolios are built from CRSP, which retains delisted firms
along with their delisting returns, and run from 1926 rather than from
whenever the survivors happened to list. Sector *funds* are not a fix either:
a fund is a managed product with fees, tracking error and its own inclusion
rules, and the set of funds still available today carries the same
survivorship problem in a different costume.

### How many portfolios? The aggregation trade-off

Fewer industries means coarser buckets, and that has a consequence which is
easy to mistake for good news. Running the same BW scaled-CAPM across all
four sets over the identical window:

| test assets | N | χ² df | R² | R²_gls | χ² p | sd(avg ret) |
|---|---|---|---|---|---|---|
| 10 industry | 10 | 7 | 0.823 | 0.752 | 0.998 | 0.071 |
| 12 industry | 12 | 9 | 0.726 | 0.726 | 0.948 | 0.085 |
| 48 industry | 48 | 45 | 0.023 | 0.037 | 0.067 | 0.162 |
| 25 size-BM | 25 | 22 | 0.818 | 0.564 | 0.076 | 0.182 |

The coarse sets look *spectacular* — R² of 0.82, GLS R² of 0.75, a χ² that
comes nowhere near rejecting. None of that is evidence the model works.

Two things are happening. First, **aggregation averages away the dispersion
you are trying to explain**: the cross-sectional standard deviation of
average returns falls from 0.162 across 48 industries to 0.071 across 10.
There is simply less left to get wrong. Second, **the fit is close to
mechanical** — you are fitting an intercept plus three λ's, four parameters,
to ten data points, leaving six residual degrees of freedom. A high R² there
is arithmetic, not evidence.

The χ² p-value of 0.998 makes the point sharpest. That is not a model fitting
beautifully; it is a test with df = 7 that cannot distinguish anything at
all. Read alongside the low dispersion, it means the test has no power.

So the 10- and 12-industry sets are useful for **description** — which
sectors behave how in good versus bad sentiment states, what the state-beta
pattern looks like across the economy — but they are weak ground for
**formal pricing tests**. The 48-industry set remains the demanding one, and
it is the only one of the four where the model is asked a hard question.

In [ ]:
def load_portfolios(path=f"{DATA_DIR}/clean_25_Portfolios_5x5.csv"):
    df = pd.read_csv(path, parse_dates=["Date"])
    df["ym"] = df["Date"].dt.year * 100 + df["Date"].dt.month
    df = df.drop(columns=["Date"]).set_index("ym")
    return df  # 25 columns, monthly returns in percent


def load_portfolios_32(path=f"{DATA_DIR}/clean_32_Portfolios_OP_INV.csv"):
    """Ken French 32 portfolios formed on Size x Operating Profitability x
    Investment (2x4x4), value-weighted returns. Alternative test-asset set
    to the 25 Size-BM portfolios, for a robustness check (recovered from the
    user's own ff32_panel.csv, long-format VW/EW returns since Jul 1963)."""
    df = pd.read_csv(path, parse_dates=["Date"])
    df["ym"] = df["Date"].dt.year * 100 + df["Date"].dt.month
    df = df.drop(columns=["Date"]).set_index("ym")
    return df  # 32 columns, monthly returns in percent


def load_portfolios_industry48(path=f"{DATA_DIR}/clean_48_Industry_Portfolios.csv"):
    """Ken French 48 Industry Portfolios, value-weighted monthly returns
    (percent), Jul 1926 - present. Missing-data codes (-99.99/-999) are
    already converted to NaN in the cleaned file (some industries, e.g.
    semiconductors, didn't exist yet in the early decades -- all 48 are
    fully populated from Jul 1969 onward). Alternative test-asset set to
    the 25 Size-BM / 32 Size-OP-INV portfolios."""
    df = pd.read_csv(path, parse_dates=["Date"])
    df["ym"] = df["Date"].dt.year * 100 + df["Date"].dt.month
    df = df.drop(columns=["Date"]).set_index("ym")
    return df  # 48 columns, monthly returns in percent (NaN where no firms)


def load_portfolios_industry12(path=f"{DATA_DIR}/clean_12_Industry_Portfolios.csv"):
    """Ken French 12 Industry Portfolios -- the closest thing in this library
    to a GICS-style sector cut: NoDur, Durbl, Manuf, Enrgy, Chems, BusEq
    (tech), Telcm, Utils, Shops, Hlth, Money (financials), Other.

    WHY THIS RATHER THAN A SCREENER. These are built from CRSP, which keeps
    delisted firms and their delisting returns, so there is no survivorship
    bias. A stock screener returns only CURRENTLY listed companies, which
    silently deletes every firm that went bankrupt, merged or delisted --
    and for sector work that is devastating, because the deletions cluster
    exactly where the interesting variation is (tech through 2000-02,
    financials through 2008). Coverage also runs from 1926 rather than
    whenever the screener's survivors happened to list.

    Fully populated -- no missing months anywhere, unlike the 48-industry
    file whose narrow buckets are empty in the early decades."""
    df = pd.read_csv(path, parse_dates=["Date"])
    df["ym"] = df["Date"].dt.year * 100 + df["Date"].dt.month
    return df.drop(columns=["Date"]).set_index("ym")


def load_portfolios_industry10(path=f"{DATA_DIR}/clean_10_Industry_Portfolios.csv"):
    """Ken French 10 Industry Portfolios: NoDur, Durbl, Manuf, Enrgy, HiTec,
    Telcm, Shops, Hlth, Utils, Other.

    NOT simply the 12-industry set with two buckets merged -- the 10-cut
    drops Chems and Money as standalone groups and folds BusEq into a
    broader HiTec. So 10 and 12 are alternative partitions of the same
    universe, not a strict nesting."""
    df = pd.read_csv(path, parse_dates=["Date"])
    df["ym"] = df["Date"].dt.year * 100 + df["Date"].dt.month
    return df.drop(columns=["Date"]).set_index("ym")


_PORT_LOADERS = {"25": load_portfolios, "32": load_portfolios_32,
                  "industry48": load_portfolios_industry48,
                  "industry12": load_portfolios_industry12,
                  "industry10": load_portfolios_industry10}

`_PORT_LOADERS` maps a short key to a loader function. Note the key is
`"industry48"`, not `"48"` — passing `"48"` raises `KeyError`.

⚠️ **Jupyter gotcha:** this dict stores *references to function objects*, captured
at the moment the dict is built. If you later redefine `load_portfolios()` in
another cell, the dict still points at the **old** function. You must re-run the
`_PORT_LOADERS = {...}` line for the change to take effect. The same applies to
`_LOADERS` further down. This is a common source of "I changed the code but the
output didn't change".

In [ ]:
# quick look at the test assets
ports = load_portfolios()
print("25 Size-BM :", ports.shape, "|", ports.index.min(), "-", ports.index.max())
print(ports.iloc[:3, :5])

---
## 2. Fama-French factors

`Mkt-RF`, `SMB`, `HML` and `RF`, monthly, in percent.

`Mkt-RF` is **already an excess return** — the market return net of the risk-free
rate. The paper's prose alternates between calling it "the market return" and
"the market excess return", but both phrases refer to this one series; their
equations use the single symbol $MktRF_t$ throughout.

`RF` is kept because the test-asset returns arrive as *total* returns and must be
converted to excess returns by subtracting it.

In [ ]:
def load_factors(path=f"{DATA_DIR}/clean_F-F_Research_Data_Factors.csv"):
    df = pd.read_csv(path, parse_dates=["Date"])
    df["ym"] = df["Date"].dt.year * 100 + df["Date"].dt.month
    return df.drop(columns=["Date"]).set_index("ym")  # Mkt-RF, SMB, HML, RF

---
## 3. Sentiment indices

The paper uses four: **BW**, **MCSI**, **CBCCI**, and **AS** (a composite of the
first three). This replication adds **PMI**, **CFNAI** and **AAII** as extra
robustness checks, and **PLS** for the Table 13 analogue.

One naming confusion worth knowing: the paper's body text (p. 215) writes
"MCSI hereafter" for the Michigan index, but its Table 3 row label and
abbreviation footnote both read "MSCI". The paper is internally inconsistent.
This code uses `"mcsi"`. (MSCI, the index provider, is unrelated.)

### 3.1 Michigan Consumer Sentiment Index (MCSI)

Survey-based measure of household optimism. It is **quarterly before 1978 and
monthly after**, so the pre-1978 portion is linearly interpolated to monthly —
exactly as the paper describes in its Section 3.1.

The interpolation is done on a true datetime axis rather than on the integer
`yyyymm` code, because interpolating on the raw integer would treat the gap from
`197012` to `197101` as 89 units instead of one month.

In [ ]:
def load_mcsi(path=f"{DATA_DIR}/mcsi_raw.csv"):
    """University of Michigan ICS: quarterly through 1977, monthly from 1978.
    Linearly interpolate to monthly, as the paper does (Section 3.1)."""
    raw = pd.read_csv(path)
    month_num = {m: i for i, m in enumerate(
        ["January", "February", "March", "April", "May", "June", "July",
         "August", "September", "October", "November", "December"], start=1)}
    raw["ym"] = raw["YYYY"] * 100 + raw["Month"].map(month_num)
    raw = raw.sort_values("ym").set_index("ym")["ICS_ALL"]
    full_index = range(raw.index.min(), raw.index.max() + 1)
    full_index = [ym for ym in range(raw.index.min() // 100 * 100 + 1, raw.index.max() + 1)
                  if 1 <= ym % 100 <= 12]
    s = raw.reindex(full_index)
    # interpolate on a true time axis (months), not the coded integer ym
    dt_index = pd.to_datetime(pd.Series(full_index).astype(str), format="%Y%m")
    s.index = dt_index
    s = s.interpolate(method="linear").bfill().ffill()
    s.index = full_index
    return s.rename("sentiment")

### 3.2 PMI, Baker-Wurgler, Conference Board, CFNAI

**BW** (Baker & Wurgler 2006) is the canonical academic sentiment index and the
paper's primary measure: the first principal component of six proxies (closed-end
fund discount, IPO count, first-day IPO returns, equity share in new issues,
dividend premium, NYSE turnover), each orthogonalized to macro variables first.
We use the maintained `SENT_ORTH` series.

**CBCCI** (Conference Board Consumer Confidence) and **PMI** (ISM Manufacturing)
come from investing.com exports. Both files needed repair — see §4 below.

**CFNAI** (Chicago Fed National Activity Index) is *not* a sentiment measure at
all; it's a business-cycle activity gauge, included only as a contrast case.

In [ ]:
def load_pmi(path=f"{DATA_DIR}/pmi_ism_raw.csv"):
    df = pd.read_csv(path).sort_values("ym").set_index("ym")["pmi"]
    return df.rename("sentiment")


def load_bw(path=f"{DATA_DIR}/bw_raw.csv"):
    """Baker & Wurgler sentiment index, orthogonalized version (SENT_ORTH),
    from the officially maintained update (github.com/BWInvestorSentimentIndex).
    This is the exact series the paper uses -- monthly since July 1965,
    already orthogonalized to macro variables, no interpolation needed."""
    df = pd.read_csv(path).rename(columns={"yearmo": "ym"}).sort_values("ym").set_index("ym")
    return df["SENT_ORTH"].dropna().rename("sentiment")


def load_cbcci(path=f"{DATA_DIR}/cbcci_raw.csv"):
    """Conference Board Consumer Confidence Index (investing.com export,
    same [Date, Time, Actual, Forecast, Previous] layout as the PMI file)."""
    df = pd.read_csv(path).sort_values("ym").set_index("ym")["cbcci"]
    return df.rename("sentiment")


def load_cfnai(path=f"{DATA_DIR}/cfnai_raw.csv"):
    """Chicago Fed National Activity Index (chicagofed.org export, main
    CFNAI series). NOT one of the paper's four indices -- a business-cycle
    activity gauge, not an investor-sentiment survey. Included as an extra
    robustness check the user asked for."""
    df = pd.read_csv(path).sort_values("ym").set_index("ym")["cfnai"]
    return df.rename("sentiment")

### 3.3 Augmented Sentiment (AS) — a real PCA on our own sample

The paper's AS index is the first principal component of BW, MCSI and CBCCI, and
they report loadings of 0.318 / 0.443 / 0.452.

**Those loadings are not reusable.** They were estimated on the paper's own
1965–2015 sample. Applying them verbatim to a different sample period would be
importing someone else's estimate rather than estimating the model. So this
function runs an actual PCA on *our* standardized data.

Two details: the inputs are z-scored first (PCA on raw levels would let the
largest-variance series dominate purely through units), and the sign of a
principal component is arbitrary, so PC1 is flipped when needed to make the
loadings positive — otherwise "high sentiment" could silently mean low sentiment.

In [ ]:
def load_as(verbose=True):
    """Augmented Sentiment (AS) index -- first principal component of
    BW, MCSI and CBCCI, via sklearn's PCA on OUR OWN standardized sample
    (NOT the paper's fixed 0.318/0.443/0.452 weights, which were fit on
    their own different sample period, so reusing them verbatim would
    be wrong)."""
    bw = load_bw()
    mcsi = load_mcsi()
    cbcci = load_cbcci()
    df = pd.DataFrame({"bw": bw, "mcsi": mcsi, "cbcci": cbcci}).dropna()
    z = (df - df.mean()) / df.std()

    pca = PCA(n_components=3)
    scores = pca.fit_transform(z.values)  # T x 3, uncorrelated components
    loadings_all = pd.DataFrame(
        pca.components_.T, index=["bw", "mcsi", "cbcci"],
        columns=["PC1", "PC2", "PC3"]
    )

    pc1_loadings = loadings_all["PC1"].copy()
    if pc1_loadings.sum() < 0:  # PCA sign is arbitrary -- flip so loadings are positive
        pc1_loadings *= -1
        scores[:, 0] *= -1

    if verbose:
        print("AS index -- PCA loadings (all 3 components):")
        print(loadings_all.round(4))
        print("Explained variance ratio:", pca.explained_variance_ratio_.round(4))
        print(f"PC1 loadings used for AS:\n{pc1_loadings.round(4)}")
        print("(paper's reported loadings, for comparison: BW=0.318, MCSI=0.443, CBCCI=0.452)")

    as_idx = pd.Series(scores[:, 0], index=z.index, name="sentiment")
    return as_idx

### 3.4 AAII, PLS, and the control variables

**AAII** is the American Association of Individual Investors' weekly survey
asking members whether they're bullish, neutral or bearish on the market over
the next six months. It is arguably the most *on-point* measure available here:
MCSI and CBCCI ask households about the **economy**, whereas AAII asks investors
directly about the **stock market**. Brown & Cliff (2004, 2005) are the standard
references. The survey is weekly, so the loader averages the weeks within each
calendar month; the default series is the bull-bear spread (bullish - bearish).
Coverage begins July 1987, which costs 211 of the 673 months in `COMMON_WINDOW`.

**PLS** (Huang, Jiang, Tu & Zhou 2015) is the Table 13 robustness measure --
sentiment extracted by partial least squares rather than principal components.
The paper doesn't construct it; it re-uses the published series.

**Controls** come from Amit Goyal's updated Welch-Goyal predictor dataset. The
paper's footnote 21 specifies the real interest rate, term premium, default
premium, inflation, and CAY. Note the paper's own equation writes
$\sum_{i=1}^{4}$ while its prose lists five -- an inconsistency in the paper.

Two things worth knowing about how `load_goyal_controls()` is built:

*`real_rate` is `Rfree - infl`.* An earlier version used `tbl - infl`, which
silently mixed an **annualised** 3-month rate with a **monthly** inflation rate.
The result was ~100x too large and effectively just tracked the nominal rate
level rather than a real rate. `Rfree` is Goyal's 30-day T-bill return, which is
what footnote 21 actually asks for. Fixing it moved the Table 2 sentiment
coefficients only slightly and changed no conclusions -- but the old variable
was not a real interest rate.

*`term_premium` is `lty - tbl`* (~20yr minus ~3mo) where the paper wants 20yr
minus 1yr. Goyal's file has no 1-year series, so this can't be fixed here.

**`load_goyal_full()`** derives all 14 standard Welch-Goyal predictors, for
testing the paper's robustness claim harder than the paper does. One trap worth
flagging: two of the 14 are **exact** linear combinations of the others --
`de = dp - ep` and `tms = lty - tbl`, both by construction -- so the 14-column
matrix has rank 12. Regressing on all 14 leaves the coefficients non-identified
(statsmodels warns and silently pseudo-inverts). `GOYAL_INDEP_COLS` holds the 12
independent ones, which is what the `"full"` control mode actually uses.

**`load_cay()`** is the Lettau-Ludvigson consumption-wealth ratio -- the
cointegrating residual between log consumption, log asset wealth and log labour
income. High cay means consumption is high relative to wealth and income, which
under consumption smoothing signals investors expect high future returns. It
matters historically: Lettau & Ludvigson (2001b) scaled the CAPM with cay,
making it the **direct methodological ancestor** of what Doukas & Han do with
sentiment. It's quarterly, and each quarter's value is assigned to its three
months as a *step function* -- not linear interpolation, which would blend in
the next quarter's value and introduce look-ahead bias in a predictive
regression. It ends 2019Q3, so using it costs ~75 months.

**`load_macro()`** is the real-activity and valuation block -- also an
extension, and the one that turned out to matter most. Three variables: the
12-month **change** in unemployment, 12-month **log growth** in housing
starts, and the **CAPE earnings yield** (1/CAPE, from Shiller's data).

Why changes and yields rather than levels: all three underlying series are
near-unit-root. The unemployment rate has monthly autocorrelation around
0.99, and CAPE trends. Regressing returns on a near-unit-root regressor over
a period when markets rose manufactures significance out of nothing -- that
is Stambaugh bias, and note the paper defends *itself* on precisely this
point (p. 220). These transformations are that same defence applied here.

Housing starts are the interesting one because they **lead**: breaking ground
is a bet on demand nine to eighteen months out, and housing is the most
interest-rate-sensitive sector there is. That makes it a genuine competitor
to sentiment, unlike CFNAI or the Philadelphia Fed's USPHCI, which are
*coincident* by construction and so are the textbook "already in the price"
case.

Two data notes. FRED serves the **current revised** vintage, not what was
known in real time, so the regression sees marginally better information than
investors had -- a footnote for the write-up. And **October 2025 has no
unemployment reading at all**: the US federal funding lapse meant the BLS
household survey was never conducted. It is left as `NaN` rather than
interpolated, because inventing a measurement that was never taken is worse
than dropping one month out of 673.

In [ ]:
def load_aaii(path=f"{DATA_DIR}/aaii_raw.csv", measure="spread"):
    """AAII Sentiment Survey -- the American Association of Individual
    Investors' weekly poll asking members whether they are bullish, neutral
    or bearish on the stock market over the next six months.

    NOT one of the paper's indices, but arguably the most on-point measure
    available here: it asks investors directly about the STOCK MARKET,
    whereas MCSI and CBCCI ask households about the ECONOMY. Brown & Cliff
    (2004, 2005) are the standard references for using it this way.

    The survey is WEEKLY (Thursdays); this loader averages the weeks within
    each calendar month, so a month is the mean of its 2-5 survey readings.

    measure= picks the series:
      "spread"  (default) bullish - bearish, the standard "bull-bear
                spread" used in the literature
      "bullish" / "bearish" / "neutral" -- the raw shares

    COVERAGE WARNING: the survey begins July 1987, so using it costs 211 of
    the 673 months in COMMON_WINDOW. Any AAII result is a shorter-sample
    check, not directly comparable to the other indices' full-window runs."""
    df = pd.read_csv(path).set_index("ym").sort_index()
    if measure not in ("spread", "bullish", "bearish", "neutral"):
        raise ValueError(f"measure must be spread/bullish/bearish/neutral; got {measure!r}")
    return df[measure].rename("sentiment")


def load_pls(path=f"{DATA_DIR}/pls_sentiment_raw.csv"):
    """PLS-based sentiment index (Huang, Jiang, Tu & Zhou, 2015, RFS) --
    the orthogonalized version, from the officially maintained update
    (Fuwei Jiang's website, through Dec 2023). This is the paper's Table 13
    robustness sentiment measure -- an externally-sourced index (built via
    partial least squares instead of principal components), not something
    Doukas & Han construct themselves; they just re-use the published
    series (their footnote 26)."""
    df = pd.read_csv(path)
    s = df.set_index("yyyymm")["PLS_SENT_ORTH"]
    s.index.name = "ym"
    return s.rename("sentiment")


def load_goyal_controls(path=f"{DATA_DIR}/goyal_controls_raw.csv"):
    """Predictive-regression control variables from Amit Goyal's updated
    Welch & Goyal (2008) predictor dataset: real interest rate, term
    premium, default premium, inflation.

    Paper's exact definitions (Doukas & Han 2021, footnote 21): real rate
    = 30-day T-bill return minus inflation; default premium = BAA - AAA
    (matches here); term premium = 20-year T-bill MINUS 1-year T-bill
    (NOT matched here -- Goyal's file has no 1-year T-bill series, only
    tbl [~3-month, ANNUALIZED] and lty [~20-year govt bond yield], so this
    uses the standard lty - tbl term spread as an approximation).

    real_rate is built as Rfree - infl, both MONTHLY rates. An earlier
    version of this file used tbl - infl, which silently mixed an
    ANNUALISED 3-month rate with a MONTHLY inflation rate -- the result was
    ~100x too large and essentially just tracked the nominal rate level.
    Rfree is Goyal's 30-day T-bill return, which is what footnote 21 asks
    for. (Fixing this moved the Table 2 sentiment coefficients only
    slightly and changed no conclusions, but the old variable was not a
    real interest rate.)

    CAY (Lettau-Ludvigson consumption-wealth ratio) is the paper's 5th
    control -- their own equation notation says Sum_{i=1}^{4} despite the
    prose listing 5, an inconsistency in the paper itself. It is NOT
    included here by default because it is quarterly and the available
    series ends long before this sample does; see load_cay() and the
    controls= argument of predictive_regression()."""
    df = pd.read_csv(path)
    df = df.set_index("yyyymm")
    df.index.name = "ym"
    return df  # real_rate, term_premium, default_premium, inflation


# The 14 standard Welch-Goyal predictors, derived below in load_goyal_full()
GOYAL_FULL_COLS = ["dp", "dy", "ep", "de", "svar", "bm", "ntis",
                   "tbl", "lty", "ltr", "tms", "dfy", "dfr", "infl"]

# Two of those 14 are EXACT linear combinations of the others, by construction:
#     de  = dp - ep          (both are log(X) - log(Index), so the Index cancels)
#     tms = lty - tbl        (the term spread IS that difference)
# Verified numerically: the 14-column matrix has rank 12. Putting all 14 in one
# regression therefore leaves the coefficients non-identified (statsmodels emits
# SingularMatrixWarning and silently pseudo-inverts). So any regression on the
# "full" set uses these 12 instead. PCA is unaffected -- it handles collinear
# inputs gracefully -- so the "pca" mode still consumes all 14.
GOYAL_INDEP_COLS = [c for c in GOYAL_FULL_COLS if c not in ("de", "tms")]


def load_goyal_full(path=f"{DATA_DIR}/goyal_predictors_raw.csv"):
    """All 14 standard Welch & Goyal (2008) return predictors, derived from
    the raw PredictorData file.

    This is an EXTENSION beyond what Doukas & Han do -- their control set is
    the four in load_goyal_controls() plus CAY. Use it to test their
    robustness claim harder than they did, not to reproduce their Table 2.

    Definitions follow Welch & Goyal:
      dp   = log(D12) - log(Index)              dividend-price ratio
      dy   = log(D12) - log(Index_{t-1})        dividend yield
      ep   = log(E12) - log(Index)              earnings-price ratio
      de   = log(D12) - log(E12)                dividend payout ratio
      svar = sum of squared daily S&P returns   stock variance
      bm   = book-to-market (DJIA)
      ntis = net equity expansion
      tbl  = 3-month T-bill rate (annualised)
      lty  = long-term (~20y) govt bond yield
      ltr  = long-term govt bond return
      tms  = lty - tbl                          term spread
      dfy  = BAA - AAA                          default yield spread
      dfr  = corpr - ltr                        default return spread
      infl = CPI inflation (monthly)

    NOTE on units: tbl/lty/tms are ANNUALISED rates while infl/ltr/dfr/svar
    are monthly. OLS is scale-invariant so this does not bias anything, but
    it does mean the fitted coefficients are not comparable across
    predictors in magnitude.

    NOTE on inflation: Welch & Goyal lag infl by one month in their own
    predictive regressions, because CPI is released with a delay. That lag
    is NOT applied here (the paper doesn't mention it either); pass
    lag_infl=True to apply it.

    'csp' (cross-sectional premium) is deliberately excluded -- Welch &
    Goyal discontinued it and it has no data over most of this sample."""
    raw = pd.read_csv(path).set_index("yyyymm").sort_index()
    # Index/D12/E12 can arrive as strings with thousands separators
    for c in ["Index", "D12", "E12"]:
        raw[c] = pd.to_numeric(raw[c].astype(str).str.replace(",", ""), errors="coerce")

    d = pd.DataFrame(index=raw.index)
    d["dp"] = np.log(raw["D12"]) - np.log(raw["Index"])
    d["dy"] = np.log(raw["D12"]) - np.log(raw["Index"].shift(1))
    d["ep"] = np.log(raw["E12"]) - np.log(raw["Index"])
    d["de"] = np.log(raw["D12"]) - np.log(raw["E12"])
    d["svar"] = raw["svar"]
    d["bm"] = raw["b/m"]
    d["ntis"] = raw["ntis"]
    d["tbl"] = raw["tbl"]
    d["lty"] = raw["lty"]
    d["ltr"] = raw["ltr"]
    d["tms"] = raw["lty"] - raw["tbl"]
    d["dfy"] = raw["BAA"] - raw["AAA"]
    d["dfr"] = raw["corpr"] - raw["ltr"]
    d["infl"] = raw["infl"]
    d.index.name = "ym"
    return d[GOYAL_FULL_COLS]


def load_cay(path=f"{DATA_DIR}/cay_raw.csv"):
    """Lettau & Ludvigson consumption-wealth ratio (cay), expanded from
    quarterly to monthly.

    cay is the cointegrating residual from the long-run relationship
    between log consumption (c), log asset wealth (a) and log labour
    income (y). A high cay means consumption is high relative to wealth and
    income, which -- under consumption smoothing -- signals that investors
    expect high future returns. It is the conditioning variable Lettau &
    Ludvigson (2001b) used to scale the CAPM, i.e. the direct methodological
    ancestor of what Doukas & Han do with sentiment.

    TWO IMPORTANT LIMITATIONS:

    1. COVERAGE. The series runs 1952Q1-2019Q3 (Lettau's current public
       vintage; the formula in its header is cay = c - 0.218a - 0.801y +
       0.441). It ends six years before this project's sample does, so
       including it as a control TRUNCATES the estimation sample to ~598 of
       the 673 months in COMMON_WINDOW. That is why it is opt-in rather
       than part of the default control set.

    2. FREQUENCY. cay is quarterly by construction (NIPA consumption and
       labour income are quarterly). Each quarter's value is assigned to all
       three of its months -- a step function, NOT linear interpolation.
       Interpolating linearly would blend in the *next* quarter's value,
       which is look-ahead bias in a predictive regression. (Contrast
       load_mcsi(), where linear interpolation is fine because MCSI is a
       contemporaneous survey level, not a predictor being used to forecast
       the following month.)

    Even the step expansion is mildly optimistic about timing: quarterly
    NIPA data is released with roughly a one-month lag, so cay for Q4 is not
    truly known until late January."""
    df = pd.read_csv(path)
    df.columns = [c.strip().lstrip("﻿") for c in df.columns]
    q = df.set_index("date")["cay"].sort_index()

    # date is coded YYYYQQ (e.g. 195104 = 1951Q4) -> expand to the 3 months
    rows = {}
    for code, val in q.items():
        year, quarter = int(code) // 100, int(code) % 100
        for m in range(3 * (quarter - 1) + 1, 3 * (quarter - 1) + 4):
            rows[year * 100 + m] = val
    s = pd.Series(rows, name="cay").sort_index()
    s.index.name = "ym"
    return s


MACRO_COLS = ["unrate_chg", "houst_gr", "cape_yield"]


def load_macro(unrate_path=f"{DATA_DIR}/unrate_raw.csv",
               houst_path=f"{DATA_DIR}/houst_raw.csv",
               shiller_path=f"{DATA_DIR}/shiller_raw.csv"):
    """Real-activity and valuation controls, for asking whether sentiment's
    predictive power survives once the real economy is held constant.

    This is an EXTENSION beyond Doukas & Han, who never test this.

    Three variables, each already transformed to be usable as a regressor:

      unrate_chg  12-month CHANGE in the unemployment rate (percentage
                  points). BLS via FRED (UNRATE), monthly from 1948.
      houst_gr    12-month LOG GROWTH in housing starts. Census/HUD via FRED
                  (HOUST), monthly from 1959, seasonally adjusted annual
                  rate. Housing LEADS the cycle -- breaking ground is a bet
                  on demand 9-18 months out, and it is the most
                  interest-rate-sensitive sector there is -- which makes it
                  a real competitor to sentiment rather than a straw man.
                  Contrast CFNAI and the Philadelphia Fed's USPHCI, which
                  are COINCIDENT by construction and so are the textbook
                  "already in the price" case.
      cape_yield  1 / CAPE, i.e. the cyclically-adjusted earnings yield.
                  Robert Shiller's data. CAPE smooths earnings over 10 years
                  and is the best-known long-horizon valuation predictor;
                  Goyal's 'ep' is the 1-year version, so this is additive
                  rather than redundant.

    WHY CHANGES AND YIELDS RATHER THAN LEVELS. All three underlying series
    are extremely persistent -- the unemployment rate has monthly
    autocorrelation near 0.99, housing starts swing cyclically, and CAPE
    trends. Regressing returns on a near-unit-root regressor over a period
    when markets rose manufactures significance out of nothing (Stambaugh
    bias). Note the paper defends ITSELF on exactly this point (p.220,
    citing Stambaugh et al. 2014 simulating 200 million equally persistent
    regressors); these transformations are the same defence applied here.

    REVISIONS CAVEAT: FRED serves the CURRENT revised vintage, not what was
    known in real time. Housing starts in particular get revised
    meaningfully, so the regression sees slightly better information than
    investors had. A footnote for the write-up, not a reason to avoid it.

    Coverage is bound by housing starts: 1959:01 plus 12 months for the
    growth rate means these are usable from 1960:01, comfortably before
    COMMON_WINDOW begins.

    THE 2025:10 GAP IS REAL AND IS LEFT AS NaN. The US federal funding lapse
    meant the BLS household survey was not conducted that month, so no
    unemployment rate exists for October 2025 -- it was never measured, as
    opposed to measured-and-lost. Interpolating would fabricate an
    observation, so this deliberately does not: unrate_chg is NaN for
    2025:10, and any regression using it drops exactly that one month out of
    673. Say so in a footnote rather than papering over it."""
    u = pd.read_csv(unrate_path).set_index("ym").sort_index()["unrate"]
    h = pd.read_csv(houst_path).set_index("ym").sort_index()["houst"]
    s = pd.read_csv(shiller_path).set_index("ym").sort_index()["cape"]

    out = pd.DataFrame(index=sorted(set(u.index) | set(h.index) | set(s.index)))
    out.index.name = "ym"
    out["unrate_chg"] = u.reindex(out.index) - u.reindex(out.index).shift(12)
    out["houst_gr"] = np.log(h.reindex(out.index)) - np.log(h.reindex(out.index).shift(12))
    out["cape_yield"] = 1.0 / s.reindex(out.index)
    return out[MACRO_COLS]

In [ ]:
_LOADERS = {"mcsi": load_mcsi, "pmi": load_pmi, "bw": load_bw,
            "cbcci": load_cbcci, "cfnai": load_cfnai, "as": load_as,
            "pls": load_pls, "aaii": load_aaii}

Same function-reference caveat as `_PORT_LOADERS`: if you redefine any loader
above, re-run the `_LOADERS = {...}` cell or the change won't propagate.

---
## 4. The common sample window

Each series has its own coverage. If every model silently uses its own maximal
available sample, the results aren't comparable — differences between indices
get confounded with differences in sample period.

Two of the raw files also arrived **broken**, in a way that only became visible
once a common window was imposed:

- **CBCCI** appeared to be missing Jan 2008 – Aug 2014, about 80 months. The cause
  was a date-format change on investing.com: older rows are stamped on the 1st of
  the *following* month with a `(Month)` annotation giving the true reference
  month, newer rows are stamped within the month itself. Parsing the release date
  naively mangles the older block. One further row, `Nov 01, 2012`, lost its
  `(Oct)` annotation entirely and had to be reassigned to October by hand.
- **PMI** had the same format transition and a one-month hole at the seam.

After repair, both cover Dec 1969 – Aug 2026 gap-free, with a single interpolated
month each (CBCCI Jan 2008, PMI Feb 2008) at the format seam.

This is worth remembering as a methodological point: **the apparent strength of
CBCCI in early runs was an artifact of a data gap that happened to exclude the
2008 financial crisis.** Restoring those months roughly halved its sentiment
coefficient. A uniform window didn't just make results comparable — it surfaced
a bug that would otherwise have gone into the write-up as a finding.

`COMMON_WINDOW` is Dec 1969 – Dec 2025. The start is bound by CBCCI/PMI, the end
by BW's maintained update. It costs the ~8 months of 2026 that other series have.

In [ ]:
COMMON_WINDOW = (196912, 202512)


def _apply_date_range(panel, date_range):
    if date_range is None:
        return panel
    start_ym, end_ym = date_range
    return panel.loc[(panel.index >= start_ym) & (panel.index <= end_ym)]

---
## 5. Building the estimation panel

`build_panel()` joins test assets, factors and sentiment on the month index,
applies the date window, and constructs the three Eq. 9 regressors.

The order of operations matters in two places:

**Drop missing portfolio months *before* standardizing sentiment.** Some industry
portfolios have no constituent firms in early decades. If sentiment were z-scored
first, its mean and standard deviation would be computed over months that get
dropped immediately afterwards.

**Standardize sentiment *within* the window.** With `date_range` set, the z-score
uses the mean and sd of the common window, so all indices are standardized over
the same period rather than over their own idiosyncratic histories.

The timing convention, once more, because it's the single easiest thing to get
wrong: `s_lag` is sentiment at $t-1$; `mkt_rf` is the market excess return at
$t$, **contemporaneous** with the portfolio return being explained; `s_mkt` is
their product. Only sentiment is lagged.

In [ ]:
def build_panel(sentiment_kind="mcsi", portfolio_kind="25", date_range=None):
    ports = _PORT_LOADERS[portfolio_kind]()
    facs = load_factors()
    sent = _LOADERS[sentiment_kind]()

    panel = ports.join(facs, how="inner").join(sent, how="inner")
    panel = panel.sort_index()
    panel = _apply_date_range(panel, date_range)
    port_cols = list(ports.columns)
    # drop any month with a missing test-asset return (e.g. an industry
    # portfolio with no firms yet in the early decades) BEFORE standardizing
    # sentiment, so the z-score isn't computed over a period we'll drop anyway
    panel = panel.dropna(subset=port_cols)
    # standardize sentiment over the estimation sample (mean 0, sd 1), as in the paper
    # -- when date_range is set, this mean/std is computed WITHIN the common
    # window, so different indices' z-scores are standardized over the same
    # period rather than each index's own idiosyncratic full history
    panel["sentiment_z"] = (panel["sentiment"] - panel["sentiment"].mean()) / panel["sentiment"].std()
    panel["s_lag"] = panel["sentiment_z"].shift(1)
    # Doukas & Han's Eq.9 timing (confirmed against the paper): the
    # regressors are LAGGED sentiment (s_lag), the CONTEMPORANEOUS market
    # return (mkt_rf, same period t as the portfolio return), and their
    # product (lagged sentiment x contemporaneous market return). Only
    # sentiment is lagged -- Mkt-RF/SMB/HML are not. This is what makes it
    # a genuine conditional CAPM: the asset's exposure to THIS period's
    # market move, scaled by sentiment known BEFORE the period started.
    panel["mkt_rf"] = panel["Mkt-RF"]
    panel["s_mkt"] = panel["s_lag"] * panel["mkt_rf"]
    panel = panel.dropna(subset=["s_lag", "s_mkt"])

    for c in port_cols:
        panel[c] = panel[c] - panel["RF"]  # excess returns
    return panel, port_cols

In [ ]:
# build one panel and inspect what came out
panel, port_cols = build_panel("mcsi", portfolio_kind="25", date_range=COMMON_WINDOW)
print("T =", len(panel), " N =", len(port_cols))
print("sample:", panel.index.min(), "-", panel.index.max())
print()
print(panel[["sentiment", "sentiment_z", "s_lag", "mkt_rf", "s_mkt", "RF"]].head())

---
## 6. First stage — time-series betas

One OLS regression per portfolio, over the **full sample**, of that portfolio's
excess return on the three factors. This yields each portfolio's factor loadings
$(b_{s,i}, b_{m,i}, b_{sm,i})$ — the exposures that the second stage will then
price.

Returns both the $N\times K$ beta matrix and the $T\times N$ residual matrix; the
residuals are needed later for the $\chi^2$ joint test.

Note the betas are estimated once over the whole sample, not rolled. The paper
does the same, and it's standard FM practice — though it does mean the betas use
information from the full period, including months after the returns they price.

In [ ]:
def first_stage_betas(panel, port_cols, factor_cols=("s_lag", "mkt_rf", "s_mkt")):
    betas = {}
    resid = {}
    for c in port_cols:
        X = sm.add_constant(panel[list(factor_cols)])
        fit = sm.OLS(panel[c], X).fit()
        betas[c] = fit.params[list(factor_cols)].values
        resid[c] = fit.resid.values
    beta_mat = pd.DataFrame(betas, index=list(factor_cols)).T  # N x K
    resid_mat = pd.DataFrame(resid, index=panel.index)          # T x N
    return beta_mat, resid_mat

In [ ]:
beta_mat, resid_mat = first_stage_betas(panel, port_cols)
print(beta_mat.round(4).head())

---
## 7. Second stage — Fama-MacBeth cross-sectional regressions

For each month $t$, regress that month's $N$ portfolio excess returns on the $N$
betas from stage one. The betas are fixed across months, so the regressor matrix
$X$ is built once outside the loop; only the dependent variable changes.

This produces a time series of $\lambda_t$ estimates. The reported risk premium is
their average, and the classical FM standard error is their sample standard
deviation divided by $\sqrt{T}$ — the paper's footnote 16. The elegance of the FM
approach is that this standard error automatically accounts for cross-sectional
correlation in the pricing errors, because each month's $\lambda_t$ is a single
draw and the variation *across* draws captures it.

The intercept `const` is worth watching closely. Since the left-hand side is
already an *excess* return, a correctly specified model should have an intercept
of **zero**. A large, significant constant means the model is explaining the
cross-section with a level shift rather than with its factors.

In [ ]:
def fama_macbeth(panel, port_cols, beta_mat):
    X = sm.add_constant(beta_mat.values)  # N x (K+1), same every month
    lambdas = []
    for t, row in panel[port_cols].iterrows():
        y = row.values
        fit = sm.OLS(y, X).fit()
        lambdas.append(fit.params)
    lam_df = pd.DataFrame(lambdas, index=panel.index,
                           columns=["const"] + list(beta_mat.columns))
    lam_mean = lam_df.mean()
    T = len(lam_df)
    se_fm = lam_df.std(ddof=1) / np.sqrt(T)
    t_fm = lam_mean / se_fm
    return lam_df, lam_mean, se_fm, t_fm

In [ ]:
lam_df, lam_mean, se_fm, t_fm = fama_macbeth(panel, port_cols, beta_mat)
print("mean lambdas:"); print(lam_mean.round(4))
print(); print("FM t-stats:"); print(t_fm.round(3))

---
## 8. The Shanken (1992) correction

The second stage regresses on **estimated** betas, not true ones. That's a
classic errors-in-variables problem, and it makes the classical FM standard
errors too small — overstating significance.

Shanken's correction inflates them by

$$\sqrt{1 + \lambda'\Sigma_f^{-1}\lambda}$$

where $\Sigma_f$ is the covariance matrix of the realized factors.

**The correction changes standard errors only — never the coefficients.** The
$\lambda$ point estimates are identical before and after; only the $t$-statistics
shrink. If you ever see the lambdas move when you apply it, something is wrong.

The multiplier applies to the factor lambdas, not to the intercept — which is why
`run()` scales only `["s_lag", "mkt_rf", "s_mkt"]`. The paper reports both:
uncorrected $t$ on top, Shanken-corrected underneath.

In [ ]:
def shanken_correction(lam_mean, panel, factor_cols=("s_lag", "mkt_rf", "s_mkt")):
    lam = lam_mean[list(factor_cols)].values
    Sigma_f = np.cov(panel[list(factor_cols)].values.T)
    mult = 1 + lam @ np.linalg.solve(Sigma_f, lam)
    return np.sqrt(mult)  # multiply classical FM SE (of the non-constant lambdas) by this

---
## 9. Goodness of fit

### 9.1 Cross-sectional $R^2$ (OLS)

Fitted expected return for portfolio $i$ is $\hat\lambda_0 + \beta_i'\hat\lambda$,
and the pricing error is $\alpha_i = \bar R_i - \hat R_i$. The $R^2$ compares
$\sum\alpha_i^2$ against the cross-sectional variance of average returns. The
adjusted version penalizes for $K$ factors — important when comparing models with
different numbers of factors on the same assets.

In [ ]:
def cross_sectional_fit(panel, port_cols, beta_mat, lam_mean, factor_cols=("s_lag", "mkt_rf", "s_mkt")):
    avg_ret = panel[port_cols].mean()
    fitted = lam_mean["const"] + beta_mat[list(factor_cols)].values @ lam_mean[list(factor_cols)].values
    fitted = pd.Series(fitted, index=port_cols)
    alpha = avg_ret - fitted
    ss_res = (alpha ** 2).sum()
    ss_tot = ((avg_ret - avg_ret.mean()) ** 2).sum()
    r2 = 1 - ss_res / ss_tot
    N, K = len(port_cols), len(factor_cols)
    r2_adj = 1 - (1 - r2) * (N - 1) / (N - K - 1)
    return avg_ret, fitted, alpha, r2, r2_adj

### 9.2 GLS $R^2$ — and a bug worth understanding

Lewellen, Nagel & Shanken (2010) argue the OLS cross-sectional $R^2$ is close to
meaningless on the 25 size-BM portfolios, for the reason given in §1: their tight
factor structure lets almost anything fit. Their proposed diagnostic weights the
pricing errors by $\Sigma^{-1}$:

$$R^2_{GLS} = 1 - \frac{\alpha'\Sigma^{-1}\alpha}{(\bar R - \bar R_{GLS}\iota)'\Sigma^{-1}(\bar R - \bar R_{GLS}\iota)}$$

**The subtlety that broke this code once:** the $\lambda$'s used to form $\alpha$
must themselves be estimated **by GLS**. An earlier version passed in the alphas
from the OLS/Fama-MacBeth second stage and merely weighted *those* by
$\Sigma^{-1}$. But GLS is by construction the estimator that *minimises*
$\alpha'\Sigma^{-1}\alpha$; with OLS lambdas the numerator isn't minimised, the
ratio can exceed 1, and the statistic goes negative. That produced GLS $R^2$ of
−0.20 to −0.44 on the 25 portfolios.

A correct GLS $R^2$ **can never be negative** when the model contains a constant,
because it nests the constant-only benchmark that defines the denominator. The
negative values were the tell. After the fix: MCSI 0.071, CBCCI 0.104, BW 0.564 —
all positive, and in line with the paper's 0.13 / 0.21 / 0.35.

A substantive finding fell out of that diagnostic. For the scaled CAPM the FM and
GLS lambdas differ *enormously*: for MCSI, $\lambda^s_m$ is −2.46 under FM but
**+0.43** under GLS — a sign flip; for CBCCI, −1.79 versus **+0.12**. Plain FF3's
lambdas barely move between weightings. So the scaled factor's price of risk is
highly sensitive to the weighting scheme, which is precisely the fragility the
GLS diagnostic exists to expose — and it lands on the paper's headline result.

One consequence to keep in mind when reading output: the GLS $R^2$ describes the
GLS estimator while the lambda table reports FM estimates. The two columns refer
to different estimators of the same model. That's the standard convention (LNS
do it, and so does the paper's Table 3), but it's worth stating explicitly.

In [ ]:
def gls_r2(panel, port_cols, beta_mat, factor_cols=("s_lag", "mkt_rf", "s_mkt")):
    """Lewellen, Nagel & Shanken (2010) GLS cross-sectional R^2:

        1 - (a' Sigma^-1 a) / ((Rbar - Rbar_gls*iota)' Sigma^-1 (Rbar - Rbar_gls*iota))

    CRITICAL: the lambdas used to form the pricing errors 'a' are re-estimated
    here by GLS cross-sectional regression -- they are NOT the OLS/Fama-MacBeth
    lambdas reported in the lambda table. This matters because the GLS estimator
    is by construction the one that MINIMISES a'Sigma^-1 a. Feeding this formula
    alphas built from OLS lambdas leaves the numerator un-minimised, so the ratio
    can exceed 1 and the statistic can come out NEGATIVE -- which a genuine GLS
    R^2 can never be, since the model nests the constant-only benchmark that
    defines the denominator. (An earlier version of this function did exactly
    that and produced GLS R^2 of -0.20 to -0.44 on the 25 portfolios.)

    Consequence worth remembering when reading the output: the GLS R^2 describes
    the fit of the GLS estimator, while the lambda table reports FM/OLS estimates.
    That is the standard convention (it is what LNS do, and what Doukas & Han
    report in their Table 3), but the two columns do refer to different
    estimators of the same model."""
    Sigma = panel[port_cols].cov().values
    Sigma_inv = np.linalg.pinv(Sigma)
    Rbar = panel[port_cols].mean().values
    N = len(port_cols)
    iota = np.ones(N)

    X = np.column_stack([iota, beta_mat[list(factor_cols)].values])  # N x (K+1)
    lam_gls = np.linalg.solve(X.T @ Sigma_inv @ X, X.T @ Sigma_inv @ Rbar)
    alpha = Rbar - X @ lam_gls

    Rbar_gls_mean = (iota @ Sigma_inv @ Rbar) / (iota @ Sigma_inv @ iota)
    dev = Rbar - Rbar_gls_mean * iota
    return 1 - (alpha @ Sigma_inv @ alpha) / (dev @ Sigma_inv @ dev)

### 9.3 $\chi^2$ joint test of the pricing errors

Tests $H_0:$ all pricing errors are jointly zero, via
$\alpha'\widehat{Cov}(\alpha)^{-1}\alpha \sim \chi^2(N-K)$, following the paper's
footnote 18.

**Read this together with the $R^2$, never alone.** A failure to reject is not
evidence a model works — it can equally mean the pricing errors are large but
imprecisely estimated. In this replication the industry portfolios don't reject
at 5% while the 25 reject at $p<0.001$, and that's the *opposite* of what the
$R^2$ values suggest. The reason is power: with $N=48$ and a diffuse covariance
structure the test can't discriminate; with $N=25$ and a tight one, it can.

In [ ]:
def chi2_joint_test(panel, port_cols, beta_mat, alpha, resid_mat, factor_cols=("s_lag", "mkt_rf", "s_mkt")):
    T = len(panel)
    N, K = len(port_cols), len(factor_cols)
    X = sm.add_constant(beta_mat[list(factor_cols)].values)  # N x (K+1)
    P = X @ np.linalg.pinv(X.T @ X) @ X.T
    I = np.eye(N)
    Sigma_eps = resid_mat.cov().values
    cov_alpha = (1 / T) * (I - P) @ Sigma_eps @ (I - P).T
    a = alpha.values
    stat = a @ np.linalg.pinv(cov_alpha) @ a
    from scipy import stats as sstats
    df = N - K
    pval = 1 - sstats.chi2.cdf(stat, df)
    return stat, df, pval

---
## 10. State beta — the paper's central mechanism

This is the heart of the paper's contribution, and arguably matters more than the
FM lambdas.

Conditional beta in a given sentiment state is
$B_i = b_{m,i} + b_{sm,i}\cdot E[s\,|\,\text{state}]$. States follow the
Lettau-Ludvigson convention the paper adopts: **good** = months where standardized
sentiment $\geq +1$ sd, **bad** = $\leq -1$ sd.

**State beta** $= B_i(\text{bad}) - B_i(\text{good})$.

The paper's claim is that while the *static* beta-return relation is flat (the
long-standing CAPM anomaly), the *state* beta-return relation slopes **upward** —
portfolios whose betas rise in bad-sentiment states earn higher average returns.
That would give a behavioural explanation for the value premium.

A sign relationship worth internalizing, because the output looks contradictory
otherwise: since $E[s|\text{bad}] < E[s|\text{good}]$, that bracket is negative
(around −2.8 with a 1 sd split), so **state beta is positive exactly when
$b_{sm}$ is negative**. A negative $\lambda_{s\_mkt}$ in the FM table and a
positive state-beta slope are therefore *the same statement*, not conflicting
ones.

Regressions use White (HC1) robust standard errors.

In [ ]:
def state_beta_analysis(panel, port_cols, beta_mat, threshold="1sd"):
    s = panel["sentiment_z"]
    if threshold == "1sd":
        good = s >= 1.0
        bad = s <= -1.0
    else:  # mean split
        good = s >= 0
        bad = s < 0
    s_good_mean = panel.loc[good, "sentiment_z"].mean()
    s_bad_mean = panel.loc[bad, "sentiment_z"].mean()

    b_m = beta_mat["mkt_rf"]
    b_sm = beta_mat["s_mkt"]
    beta_good = b_m + b_sm * s_good_mean
    beta_bad = b_m + b_sm * s_bad_mean
    state_beta = beta_bad - beta_good  # paper's convention: bad - good

    ret_good = panel.loc[good, port_cols].mean()
    ret_bad = panel.loc[bad, port_cols].mean()
    ret_all = panel[port_cols].mean()
    ret_market_beta = beta_mat["mkt_rf"]  # static market beta proxy (b_m only)

    def slope_reg(x, y):
        X = sm.add_constant(x.values)
        fit = sm.OLS(y.values, X).fit(cov_type="HC1")
        return fit.params[1], fit.tvalues[1], fit.rsquared

    out = {}
    out["state_beta_vs_avg_return"] = slope_reg(state_beta, ret_all)
    out["bad_beta_vs_bad_return"] = slope_reg(beta_bad, ret_bad)
    out["good_beta_vs_good_return"] = slope_reg(beta_good, ret_good)
    out["static_beta_vs_avg_return"] = slope_reg(ret_market_beta, ret_all)
    table = pd.DataFrame({
        "market_beta": b_m, "state_beta": state_beta,
        "beta_good_state": beta_good, "beta_bad_state": beta_bad,
        "ret_all": ret_all, "ret_good": ret_good, "ret_bad": ret_bad,
    })
    return out, table

---
## 11. `run()` — the Eq. 9 driver

Chains everything above: build panel → first stage → Fama-MacBeth → Shanken →
fit statistics → state beta. This is the Table 3 analogue.

Everything is returned in a dict, so you can dig into `beta_mat`, `panel`, or
`state_table` afterwards rather than only reading the printed summary.

In [ ]:
def run(sentiment_kind="mcsi", threshold="1sd", portfolio_kind="25", date_range=None):
    panel, port_cols = build_panel(sentiment_kind, portfolio_kind, date_range)
    beta_mat, resid_mat = first_stage_betas(panel, port_cols)
    lam_df, lam_mean, se_fm, t_fm = fama_macbeth(panel, port_cols, beta_mat)
    shanken_mult = shanken_correction(lam_mean, panel)
    se_shanken = se_fm.copy()
    se_shanken[["s_lag", "mkt_rf", "s_mkt"]] *= shanken_mult
    t_shanken = lam_mean / se_shanken

    avg_ret, fitted, alpha, r2, r2_adj = cross_sectional_fit(panel, port_cols, beta_mat, lam_mean)
    r2gls = gls_r2(panel, port_cols, beta_mat)
    chi2, chi2_df, chi2_p = chi2_joint_test(panel, port_cols, beta_mat, alpha, resid_mat)
    state_out, state_table = state_beta_analysis(panel, port_cols, beta_mat, threshold)

    print(f"\n{'='*78}\nSENTIMENT-SCALED CAPM  --  sentiment = {sentiment_kind.upper()}"
          f"  ({threshold} good/bad split, {portfolio_kind}-portfolio test assets)\n{'='*78}")
    print(f"Sample: {panel.index.min()} - {panel.index.max()}  (T = {len(panel)} months, N = {len(port_cols)} portfolios)\n")

    print("-- Fama-MacBeth cross-sectional regression (Table 3 analogue) --")
    tbl = pd.DataFrame({
        "lambda": lam_mean, "t_FM": t_fm, "SE_FM": se_fm,
        "t_Shanken": t_shanken, "SE_Shanken": se_shanken,
    })
    print(tbl.round(4))
    print(f"\nR^2 (unadjusted): {r2:.4f}   R^2 (adjusted): {r2_adj:.4f}   R^2 (GLS): {r2gls:.4f}")
    print(f"Chi2 joint pricing-error test: {chi2:.2f}  (df={chi2_df}, p={chi2_p:.4f})")

    print("\n-- State-beta security market line (Table 5-7 analogue) --")
    for k, (b, t, r2_) in state_out.items():
        print(f"  {k:32s}  slope={b:8.4f}   t={t:7.2f}   R2={r2_:.3f}")

    return {
        "panel": panel, "port_cols": port_cols, "beta_mat": beta_mat,
        "lambda_table": tbl, "r2": r2, "r2_adj": r2_adj, "r2_gls": r2gls,
        "chi2": (chi2, chi2_df, chi2_p), "state_out": state_out,
        "state_table": state_table,
    }

In [ ]:
res = run("mcsi", portfolio_kind="25", date_range=COMMON_WINDOW)

---
## 12. Sentiment-scaled FF3 — Table 11

A different equation from Eq. 9, not an extension of it. The factors are
$s_{t-1}MktRF_t$, $s_{t-1}SMB_t$, $s_{t-1}HML_t$ — **all three are interactions**,
and there is no bare sentiment level term and no bare market term.

That absence turns out to matter. In earlier runs CBCCI looked strong under Eq. 9
specifically because of its bare `s_lag` level term; moving to this spec erased
the advantage entirely, which located the effect in the level term rather than in
the beta-scaling mechanism the paper theorizes.

SMB and HML are contemporaneous here, same as `mkt_rf` — only sentiment is lagged.

In [ ]:
def build_panel_ff3(sentiment_kind="mcsi", portfolio_kind="25", date_range=None):
    """Panel for the paper's Table 11 spec: sentiment-scaled FF3, i.e. the
    first-stage regressors are s_{t-1}*MktRF_t, s_{t-1}*SMB_t, s_{t-1}*HML_t
    -- NOT a plain s_{t-1} level term (unlike the base scaled-CAPM Eq.9).
    Only sentiment is lagged; SMB/HML are contemporaneous, same timing as
    MktRF in build_panel() (see that function's docstring)."""
    panel, port_cols = build_panel(sentiment_kind, portfolio_kind, date_range)
    panel["s_smb"] = panel["s_lag"] * panel["SMB"]
    panel["s_hml"] = panel["s_lag"] * panel["HML"]
    return panel, port_cols


def run_ff3(sentiment_kind="mcsi", portfolio_kind="25", date_range=None):
    """Table 11 analogue: E_t(R_i,t+1) = rf + b^s_i,m*lam^s_m + b^s_i,smb*lam^s_smb
    + b^s_i,hml*lam^s_hml, on the 25 Size-BM portfolios (paper tests all four
    sentiment indices against this spec)."""
    factor_cols = ("s_mkt", "s_smb", "s_hml")
    panel, port_cols = build_panel_ff3(sentiment_kind, portfolio_kind, date_range)
    beta_mat, resid_mat = first_stage_betas(panel, port_cols, factor_cols)
    lam_df, lam_mean, se_fm, t_fm = fama_macbeth(panel, port_cols, beta_mat)
    shanken_mult = shanken_correction(lam_mean, panel, factor_cols)
    se_shanken = se_fm.copy()
    se_shanken[list(factor_cols)] *= shanken_mult
    t_shanken = lam_mean / se_shanken

    avg_ret, fitted, alpha, r2, r2_adj = cross_sectional_fit(
        panel, port_cols, beta_mat, lam_mean, factor_cols)
    r2gls = gls_r2(panel, port_cols, beta_mat, factor_cols)

    print(f"\n{'='*78}\nSENTIMENT-SCALED FF3 (Table 11 analogue)  --  sentiment = {sentiment_kind.upper()}"
          f"  ({portfolio_kind}-portfolio test assets)\n{'='*78}")
    print(f"Sample: {panel.index.min()} - {panel.index.max()}  (T = {len(panel)} months, N = {len(port_cols)} portfolios)\n")
    tbl = pd.DataFrame({
        "lambda": lam_mean, "t_FM": t_fm, "SE_FM": se_fm,
        "t_Shanken": t_shanken, "SE_Shanken": se_shanken,
    })
    print(tbl.round(4))
    print(f"\nR^2 (unadjusted): {r2:.4f}   R^2 (adjusted): {r2_adj:.4f}   R^2 (GLS): {r2gls:.4f}")

    return {
        "panel": panel, "port_cols": port_cols, "beta_mat": beta_mat,
        "lambda_table": tbl, "r2": r2, "r2_adj": r2_adj, "r2_gls": r2gls,
    }

---
## 13. Plain FF3 — the unscaled baseline

Standard Fama-MacBeth on $MktRF$, $SMB$, $HML$ with no sentiment anywhere. This
is the benchmark that makes the sentiment results interpretable: without it,
"the scaled CAPM gets $R^2$ of 0.54" is a number with nothing to compare against.

`sentiment_kind` here only pins the sample window to match the sentiment runs, so
the comparison is on identical months. Both models have $K=3$, so $\chi^2$ degrees
of freedom and the adjusted-$R^2$ penalty are identical too — a clean head-to-head.

Note this function reads the **raw** `Mkt-RF`/`SMB`/`HML` columns, deliberately
bypassing the lowercase `mkt_rf` that `build_panel()` creates. That keeps it a
genuine literature-standard baseline regardless of what happens to the scaled spec.

In [ ]:
def run_ff3_plain(sentiment_kind="mcsi", portfolio_kind="25", date_range=None):
    """The 'usual' Fama-French 3-factor Fama-MacBeth test -- NOT scaled by
    sentiment (contrast with run_ff3(), which is the sentiment-SCALED FF3,
    Table 11 analogue). This is a baseline: plain Mkt-RF/SMB/HML betas
    priced via standard two-pass Fama-MacBeth, sentiment plays no role in
    the regression itself. sentiment_kind only restricts the sample to the
    same overlap window used by the sentiment-scaled tests, so R^2/chi2
    are directly comparable across models on an apples-to-apples sample."""
    factor_cols = ("Mkt-RF", "SMB", "HML")
    panel, port_cols = build_panel(sentiment_kind, portfolio_kind, date_range)
    beta_mat, resid_mat = first_stage_betas(panel, port_cols, factor_cols)
    lam_df, lam_mean, se_fm, t_fm = fama_macbeth(panel, port_cols, beta_mat)
    shanken_mult = shanken_correction(lam_mean, panel, factor_cols)
    se_shanken = se_fm.copy()
    se_shanken[list(factor_cols)] *= shanken_mult
    t_shanken = lam_mean / se_shanken

    avg_ret, fitted, alpha, r2, r2_adj = cross_sectional_fit(
        panel, port_cols, beta_mat, lam_mean, factor_cols)
    r2gls = gls_r2(panel, port_cols, beta_mat, factor_cols)
    chi2, chi2_df, chi2_p = chi2_joint_test(panel, port_cols, beta_mat, alpha, resid_mat, factor_cols)

    print(f"\n{'='*78}\nPLAIN (unscaled) FAMA-FRENCH 3-FACTOR MODEL -- baseline"
          f"  ({portfolio_kind}-portfolio test assets, sample matched to {sentiment_kind.upper()})\n{'='*78}")
    print(f"Sample: {panel.index.min()} - {panel.index.max()}  (T = {len(panel)} months, N = {len(port_cols)} portfolios)\n")
    tbl = pd.DataFrame({
        "lambda": lam_mean, "t_FM": t_fm, "SE_FM": se_fm,
        "t_Shanken": t_shanken, "SE_Shanken": se_shanken,
    })
    print(tbl.round(4))
    print(f"\nR^2 (unadjusted): {r2:.4f}   R^2 (adjusted): {r2_adj:.4f}   R^2 (GLS): {r2gls:.4f}")
    print(f"Chi2 joint pricing-error test: {chi2:.2f}  (df={chi2_df}, p={chi2_p:.4f})")

    return {
        "panel": panel, "port_cols": port_cols, "beta_mat": beta_mat,
        "lambda_table": tbl, "r2": r2, "r2_adj": r2_adj, "r2_gls": r2gls,
        "chi2": (chi2, chi2_df, chi2_p),
    }

---
## 14. Anomaly portfolios — Tables 8/9 (partial)

**This is a partial replication and the substitutions are substantial.** The paper
uses eight Stambaugh-Yu-Yuan (2012) anomalies built from CRSP/Compustat via WRDS:
asset growth, net operating assets, net stock issues, total accruals, composite
equity issuance, investment-to-assets, return on equity, failure probability.

Without WRDS access, the four closest public analogues from Ken French are
substituted — net share issues, investment, accruals, and operating profitability
standing in for ROE. So this is **4 of 8 anomalies, with different variable
definitions**. Treat these results as indicative only.

Long/short leg directions follow the standard convention that the side theory
predicts should earn more is "long": low NI/INV/AC, but *high* OP.

In [ ]:
_ANOMALY_CODES = ["NI", "INV", "AC", "OP"]
_ANOMALY_LONG_SHORT = {
    "NI": ("Dec1", "Dec10"), "INV": ("Dec1", "Dec10"),
    "AC": ("Dec1", "Dec10"), "OP": ("Dec10", "Dec1"),
}


def load_anomaly_deciles(code):
    path = f"{DATA_DIR}/clean_{code}_deciles.csv"
    df = pd.read_csv(path, parse_dates=["Date"])
    df["ym"] = df["Date"].dt.year * 100 + df["Date"].dt.month
    df = df.drop(columns=["Date"]).set_index("ym")
    df.columns = [f"{code}_{c}" for c in df.columns]
    return df  # 10 columns (Dec1..Dec10), monthly returns in percent


def load_portfolios_anomaly_pool():
    """Pooled deciles across the 4 available anomalies (40 portfolios total
    -- the paper pools 8 anomalies x 10 deciles = 80)."""
    parts = [load_anomaly_deciles(c) for c in _ANOMALY_CODES]
    df = parts[0]
    for p in parts[1:]:
        df = df.join(p, how="inner")
    return df


_PORT_LOADERS["anomaly_pool"] = load_portfolios_anomaly_pool


def _build_single_anomaly_panel(code, sentiment_kind="as"):
    ports = load_anomaly_deciles(code)
    port_cols = list(ports.columns)
    facs = load_factors()
    sent = _LOADERS[sentiment_kind]()
    panel = ports.join(facs, how="inner").join(sent, how="inner").sort_index()
    panel["sentiment_z"] = (panel["sentiment"] - panel["sentiment"].mean()) / panel["sentiment"].std()
    panel["s_lag"] = panel["sentiment_z"].shift(1)
    panel["mkt_rf"] = panel["Mkt-RF"]
    panel["s_mkt"] = panel["s_lag"] * panel["mkt_rf"]
    panel = panel.dropna(subset=["s_lag", "s_mkt"])
    for c in port_cols:
        panel[c] = panel[c] - panel["RF"]
    return panel, port_cols


def table8_state_beta(sentiment_kind="as", threshold="1sd"):
    """Table 8 Panel B analogue: state-beta regression (Tables 6/7
    methodology) run separately on each anomaly's own 10 deciles."""
    rows = {}
    for code in _ANOMALY_CODES:
        panel, port_cols = _build_single_anomaly_panel(code, sentiment_kind)
        beta_mat, _ = first_stage_betas(panel, port_cols)
        state_out, _ = state_beta_analysis(panel, port_cols, beta_mat, threshold)
        b, t, r2 = state_out["state_beta_vs_avg_return"]
        rows[code] = {"b": b, "t": t, "R2": r2}
    return pd.DataFrame(rows).T


def table9_long_short(sentiment_kind="as", threshold="1sd"):
    """Table 9 analogue: returns and conditional betas of the long and
    short legs of each anomaly, split by good/bad sentiment state."""
    rows = []
    for code in _ANOMALY_CODES:
        panel, port_cols = _build_single_anomaly_panel(code, sentiment_kind)
        long_suffix, short_suffix = _ANOMALY_LONG_SHORT[code]
        long_col, short_col = f"{code}_{long_suffix}", f"{code}_{short_suffix}"

        beta_mat, _ = first_stage_betas(panel, port_cols)
        s = panel["sentiment_z"]
        if threshold == "1sd":
            good, bad = s >= 1.0, s <= -1.0
        else:
            good, bad = s >= 0, s < 0
        s_good_mean = panel.loc[good, "sentiment_z"].mean()
        s_bad_mean = panel.loc[bad, "sentiment_z"].mean()

        def cond_beta(col):
            b_m, b_sm = beta_mat.loc[col, "mkt_rf"], beta_mat.loc[col, "s_mkt"]
            return b_m + b_sm * s_good_mean, b_m + b_sm * s_bad_mean  # (good, bad)

        long_beta_good, long_beta_bad = cond_beta(long_col)
        short_beta_good, short_beta_bad = cond_beta(short_col)

        long_ret_good, long_ret_bad = panel.loc[good, long_col].mean(), panel.loc[bad, long_col].mean()
        short_ret_good, short_ret_bad = panel.loc[good, short_col].mean(), panel.loc[bad, short_col].mean()
        ls_ret_good = long_ret_good - short_ret_good
        ls_ret_bad = long_ret_bad - short_ret_bad

        rows.append({
            "anomaly": code,
            "long_ret_good": long_ret_good, "long_ret_bad": long_ret_bad,
            "short_ret_good": short_ret_good, "short_ret_bad": short_ret_bad,
            "long_short_good": ls_ret_good, "long_short_bad": ls_ret_bad,
            "long_beta_good": long_beta_good, "long_beta_bad": long_beta_bad,
            "short_beta_good": short_beta_good, "short_beta_bad": short_beta_bad,
        })
    return pd.DataFrame(rows).set_index("anomaly")


def run_anomaly_tables(sentiment_kind="as", threshold="1sd"):
    """Driver for the partial Table 8/9 replication (see module-level note
    above on what's substituted and why)."""
    print(f"\n{'='*78}\nANOMALY PORTFOLIOS -- PARTIAL Table 8/9 analogue  --  sentiment = {sentiment_kind.upper()}"
          f"\n(4 of the paper's 8 anomalies: NI, INV, AC, OP-as-ROE -- see module docstring)\n{'='*78}")

    print("\n-- Table 8 Panel B analogue: per-anomaly state-beta regression --")
    t8 = table8_state_beta(sentiment_kind, threshold)
    print(t8.round(4))

    print("\n-- Table 8 Panel C analogue: pooled 40-portfolio FMB cross-section --")
    pooled = run(sentiment_kind, threshold=threshold, portfolio_kind="anomaly_pool")

    print("\n-- Table 9 analogue: long/short leg returns & conditional betas by state --")
    t9 = table9_long_short(sentiment_kind, threshold)
    print(t9.round(4))

    return {"table8_state_beta": t8, "pooled_fmb": pooled, "table9": t9}

---
## 15. Predictive regression — Tables 2 and 13 Panel A

A different question from everything above. Rather than explaining the
*cross-section*, this asks whether sentiment predicts the *time series* of
next-month market excess returns:

$$MktRF_{t+1} = a + b\cdot Sentiment_t + \sum\alpha_i Control_{i,t} + e_t$$

Newey-West HAC standard errors with 3 lags, since overlapping monthly return data
is serially correlated.

The timing is expressed by pushing the target forward (`shift(-horizon)`) rather
than lagging the predictors, but it's the same structure: predictors at $t$,
outcome at $t+1$.

`controls=` selects the control set. **`"paper"` is the default**, so Table 2
stays a faithful replication:

| mode | controls used |
|---|---|
| `"paper"` | the paper's four: real rate, term premium, default premium, inflation |
| `"full"` | the 12 independent Welch-Goyal predictors |
| `"pca"` | first `n_pc` principal components of all 14 (z-scored first) |

`add_macro=True` appends the real-activity/valuation block; `add_cay=True`
appends the consumption-wealth ratio (which truncates the sample to ~597
months, so treat it as a short-sample check). The macro block costs nothing
in sample length beyond the single missing 2025:10 month.

**What `add_macro=True` shows -- the sharpest result in this project.**
Sentiment's predictive coefficient largely collapses once the macro block is
included:

| index | paper's 4 controls | + macro block |
|---|---|---|
| BW | -0.219 (t = -1.13) | **-0.002 (t = -0.01)** |
| CBCCI | -0.349 (t = -1.41) | -0.171 (t = -0.69) |
| AAII | -0.382 (t = -1.73) | -0.118 (t = -0.51) |

Adding the three one at a time identifies the culprit, and it is **not** real
activity:

| added alone | BW | CBCCI | AAII |
|---|---|---|---|
| housing starts growth | -0.213 | -0.413 | -0.390 |
| unemployment change | -0.173 | -0.253 | -0.365 |
| **CAPE yield** | **-0.083** | **-0.244** | **-0.148** |

Housing does essentially nothing -- it even *strengthens* the sentiment
coefficient for CBCCI and AAII. Unemployment absorbs a little. The CAPE
earnings yield does most of the work by itself.

The correlations say why: sentiment runs at -0.25 to -0.37 against the CAPE
yield across every index. High sentiment coincides with an expensive market.
So a large part of "sentiment predicts low future returns" may simply be
"expensive markets predict low future returns" -- a much older and far better
established result. That is a real challenge to the behavioural reading, and
a test the paper never runs.

The `"pca"` mode exists because the Welch-Goyal predictors are badly collinear;
principal components keep most of the macro information while leaving the
regression well conditioned. Welch and Goyal themselves evaluate their
predictors **one at a time** rather than jointly, for the same reason.

In [ ]:
def predictive_regression(sentiment_kind="mcsi", with_controls=True, horizon=1,
                          date_range=None, controls="paper", n_pc=3,
                          add_cay=False, add_macro=False):
    """Table 2 / Table 13 Panel A analogue:
    MktRF_{t+1} = a + b*Sentiment_t + sum(alpha_i * Controls_t) + e_t
    (HAC/Newey-West SEs, 3 lags).

    controls= selects WHICH control set (ignored when with_controls=False):

      "paper" (default) -- the four in load_goyal_controls(): real_rate,
          term_premium, default_premium, inflation. This is what reproduces
          Doukas & Han's Table 2 Panel B, so it stays the default.

      "full"  -- the Welch-Goyal predictors, as the 12 linearly independent
          ones (GOYAL_INDEP_COLS: 'de' and 'tms' are dropped because they
          are EXACT combinations of others -- see the note there). An
          EXTENSION beyond the paper: a much more demanding test of their
          claim that sentiment's predictive power isn't macro information in
          disguise. Caveat: even these 12 are heavily collinear (dp/dy/ep
          especially), so individual coefficients are unstable and hard to
          read, even though the sentiment coefficient and R^2 remain
          meaningful. Welch & Goyal themselves evaluate their predictors
          ONE AT A TIME rather than jointly, for exactly this reason.

      "pca"   -- the first n_pc principal components of those 14, z-scored
          first. The standard remedy for the collinearity above: it keeps
          most of the macro information while leaving the regression well
          conditioned. n_pc=3 by default.

    add_macro=True appends the real-activity/valuation block from
    load_macro(): the 12-month change in unemployment, 12-month growth in
    housing starts, and the CAPE earnings yield. This is the test of whether
    sentiment survives controlling for the real economy -- a question the
    paper never asks. Costs nothing in sample length (the block is available
    from 1960:01), except the single 2025:10 month that has no unemployment
    reading; see load_macro().

    add_cay=True appends the Lettau-Ludvigson consumption-wealth ratio.
    WARNING: this TRUNCATES the sample hard -- cay ends in 2003Q1 in the
    bundled file (2019Q3 even in Lettau's newest public series), against a
    sample that otherwise runs to 2025:12. Treat any cay specification as a
    short-sample robustness check, never as the headline result.

    date_range=(start_ym, end_ym) restricts the sample -- controls included
    -- to a common window (e.g. COMMON_WINDOW) before standardizing
    sentiment, so different indices are compared on an identical sample."""
    facs = load_factors()
    sent = _LOADERS[sentiment_kind]()
    panel = facs.join(sent, how="inner").sort_index()
    control_cols = []

    if with_controls:
        if controls == "paper":
            panel = panel.join(load_goyal_controls(), how="inner")
            control_cols = ["real_rate", "term_premium", "default_premium", "inflation"]
        elif controls in ("full", "pca"):
            panel = panel.join(load_goyal_full(), how="inner")
            # "full" regresses on the 12 independent predictors (de/tms are exact
            # combinations of others); "pca" keeps all 14 since PCA is untroubled
            # by collinearity and the components are built from the full set.
            control_cols = list(GOYAL_FULL_COLS) if controls == "pca" else list(GOYAL_INDEP_COLS)
        else:
            raise ValueError(f"controls must be 'paper', 'full' or 'pca'; got {controls!r}")

    if add_macro:
        panel = panel.join(load_macro(), how="inner")
        control_cols = control_cols + list(MACRO_COLS)

    if add_cay:
        panel = panel.join(load_cay(), how="inner")
        control_cols = control_cols + ["cay"]

    panel = _apply_date_range(panel, date_range)
    panel["sentiment_z"] = (panel["sentiment"] - panel["sentiment"].mean()) / panel["sentiment"].std()
    panel["mkt_rf_fwd"] = panel["Mkt-RF"].shift(-horizon)

    cols = ["sentiment_z"] + control_cols
    reg_df = panel.dropna(subset=["mkt_rf_fwd"] + cols).copy()

    if with_controls and controls == "pca":
        # collapse the 14 collinear predictors into n_pc orthogonal components.
        # standardize first, else the annualised-rate columns dominate purely
        # through their units (see the units note in load_goyal_full).
        macro = reg_df[list(GOYAL_FULL_COLS)]
        z = (macro - macro.mean()) / macro.std()
        pcs = PCA(n_components=n_pc).fit_transform(z.values)
        pc_cols = [f"macro_pc{i+1}" for i in range(n_pc)]
        for i, c in enumerate(pc_cols):
            reg_df[c] = pcs[:, i]
        extra = (list(MACRO_COLS) if add_macro else []) + (["cay"] if add_cay else [])
        cols = ["sentiment_z"] + pc_cols + extra

    X = sm.add_constant(reg_df[cols])
    y = reg_df["mkt_rf_fwd"]
    fit = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 3})
    return fit, reg_df


def run_table13(sentiment_kind="pls", threshold="1sd", portfolio_kind="25", date_range=None,
                controls="paper", add_cay=False, add_macro=False):
    """Table 13 analogue: PLS-sentiment robustness check.
    Panel A: predictive regression of next-month market excess return on
    sentiment + macro controls. Panel B: conditional CAPM scaled by
    sentiment (identical spec to Eq.9/Table 3 -- just reuses run()).
    controls=/add_cay= are passed through to predictive_regression()."""
    print(f"\n{'='*78}\nTABLE 13 analogue  --  sentiment = {sentiment_kind.upper()}\n{'='*78}")
    print(f"\n-- Panel A analogue: predictive regression (controls={controls}, "
          f"cay={add_cay}, macro={add_macro}) --")
    fit, reg_df = predictive_regression(sentiment_kind, with_controls=True, date_range=date_range,
                                        controls=controls, add_cay=add_cay, add_macro=add_macro)
    coef = fit.params["sentiment_z"]
    tstat = fit.tvalues["sentiment_z"]
    print(f"  beta_sentiment = {coef:.4f}   t (HAC) = {tstat:.3f}   (T = {len(reg_df)})")
    print(fit.summary().tables[1])

    print("\n-- Panel B analogue: conditional CAPM scaled by sentiment (Eq.9/Table 3 spec) --")
    panelB = run(sentiment_kind, threshold=threshold, portfolio_kind=portfolio_kind, date_range=date_range)

    return {"panelA_fit": fit, "panelB": panelB}


def run_table2(date_range=None, controls="paper", add_cay=False, add_macro=False, n_pc=3):
    """Table 2 analogue: predictive regression of next-month market excess
    return on lagged sentiment, Panel A (univariate, Eq.11) and Panel B
    (with controls), for the paper's four indices.

    controls= / add_cay= / n_pc= pass through to predictive_regression().
    Defaults reproduce the paper; controls="full" or "pca" run the extended
    macro control sets, and add_cay=True adds the consumption-wealth ratio
    at the cost of a much shorter sample. See that function's docstring."""
    print(f"\n{'='*78}\nTABLE 2 analogue -- sentiment predicts next-month market excess return\n{'='*78}")
    rows_a, rows_b = {}, {}
    for kind in ["bw", "mcsi", "cbcci", "as"]:
        fit_a, df_a = predictive_regression(kind, with_controls=False, date_range=date_range,
                                            add_cay=False)
        rows_a[kind.upper()] = {
            "beta": fit_a.params["sentiment_z"], "t": fit_a.tvalues["sentiment_z"],
            "R2_pct": 100 * fit_a.rsquared, "T": len(df_a),
        }
        fit_b, df_b = predictive_regression(kind, with_controls=True, date_range=date_range,
                                            controls=controls, n_pc=n_pc,
                                            add_cay=add_cay, add_macro=add_macro)
        rows_b[kind.upper()] = {
            "beta": fit_b.params["sentiment_z"], "t": fit_b.tvalues["sentiment_z"],
            "R2_pct": 100 * fit_b.rsquared, "T": len(df_b),
        }
    panelA = pd.DataFrame(rows_a).T
    panelB = pd.DataFrame(rows_b).T
    print("\n-- Panel A: univariate (Eq.11), HAC/Newey-West(3) SEs --")
    print(panelA.round(4))
    label = {"paper": "real rate, term premium[approx], default premium, inflation",
             "full": f"{len(GOYAL_INDEP_COLS)} independent Welch-Goyal predictors",
             "pca": f"first {n_pc} PCs of the {len(GOYAL_FULL_COLS)} Welch-Goyal predictors"}[controls]
    if add_macro:
        label += " + macro (unemployment, housing, CAPE yield)"
    if add_cay:
        label += " + cay (SHORT SAMPLE)"
    print(f"\n-- Panel B: with controls ({label}) --")
    print(panelB.round(4))
    return {"panelA": panelA, "panelB": panelB}

---
## 16. `run_all_sentiments()` — the comparison driver

Runs Eq. 9 across the five individual sentiment indices on one asset set and one
window, then prints a summary table. Defaults to `COMMON_WINDOW` because an
apples-to-apples comparison is the entire point of the function.

**AS is deliberately excluded.** It's constructed *from* BW, MCSI and CBCCI, so
including it alongside those three would double-count the same information.

---
## 16b. Omnibus tests — error bars on the $R^2$, and misspecification-robust SEs

Everything above reports an $R^2$ with **no standard error**, and
$t$-statistics that assume the model is **correctly specified**. Both
assumptions carry weight, and both can be checked against an independent
reference implementation: Kroencke & Thimme's `omnibus()`, a single dispatcher
for roughly 70 cross-sectional tests (`05_Omnibus/`). *Their header asks that
the paper be cited whenever the code is used.*

Two results matter.

**The $R^2$ is far less precise than it looks.** For BW on the 25 portfolios,
method 5.14 gives $R^2 = 0.8177$ with a standard error of **0.1605** — a 95%
interval of roughly $[0.50, 1.00]$. So comparing BW's 0.82 against MCSI's 0.54
is comparing two point estimates whose intervals overlap heavily. Every $R^2$
comparison in this notebook should be read with that width in mind.

**Misspecification-robust standard errors change the picture.** Ordinary
Fama-MacBeth and Shanken SEs are valid only if the model is true. Given the GLS
$R^2$, the $\chi^2$ rejections and the sign flip in $\lambda_{s\_mkt}$ under
reweighting, that is not safe here. Method 3.07 (Kan/Robotti/Shanken) drops
that assumption — and $\lambda_{s\_lag}$'s $t$ falls from 2.31 to **2.14**
while $\lambda_{s\_mkt}$'s goes from −1.91 to **−1.45**.

It also **validates our own code**. Their `R2i` (0.8177) and `R2i_gls`
(0.5643) match `run()` to four decimals, as do all three factor lambdas — an
independent confirmation, and in particular a confirmation of the GLS $R^2$ fix
described in §9.2. The one difference is the intercept's $t$: their 3.03
applies the Shanken multiplier to the constant, `run()` deliberately does not.

### Running individual methods

`omnibus_method(n)` runs any one of the ~70 tests on this project's panel.
The catalogue, from `Omnibus.py`'s own header:

| range | what |
|---|---|
| 1.01–1.06 | preliminary — are the betas all zero, or all equal? |
| 2.01–2.10 | cross-sectional regression **without** intercept |
| 3.01–3.10 | cross-sectional regression **with** intercept (3.06 Giglio/Xiu, 3.07 KRS robust) |
| 4.01–4.21 | pricing-error tests without intercept (GRS, χ², FAR, …) |
| 5.05–5.18 | pricing-error tests with intercept (5.11–5.14 the KRS R² tests) |
| 6.01–6.04 | SDF loadings — what `sdf_gmm.py` reimplements |

**Write the number with two decimals.** `omnibus()` dispatches on an exact
float comparison (`if method == 1.01:`), so `1.1` is not `1.01` and `3.7` is
not `3.07`. When nothing matches, their code does **not** raise — it returns
the base dict with none of the method-specific keys, so a typo looks like a
successful run that mysteriously has no test statistic. `omnibus_method()`
raises instead.

Methods known to fail inside their own code on this data: 5.13. Their header
also flags 2.09/3.09 and 6.03/6.04.

One caution on key names: `omnibus()` returns `R2` (no intercept) *and* `R2i`
(with intercept), and they differ a lot — 0.5862 versus 0.8177 here. Method
5.14's standard error belongs to `R2i`. Pairing the SE with the wrong one is an
easy mistake.

In [ ]:
OMNIBUS_DIR = "05_Omnibus"


def _load_omnibus(omnibus_dir=OMNIBUS_DIR):
    """Import Kroencke & Thimme's omnibus() from 05_Omnibus/.

    Their code imports its six helpers (nw, hac_var, linchi2, cdfchic,
    block_bootstrap, FMB_coefficients) as TOP-LEVEL modules, so the folder
    has to be on sys.path -- importing the file by path alone fails."""
    import sys
    d = os.path.abspath(omnibus_dir)
    if not os.path.isdir(d):
        raise FileNotFoundError(
            f"{d} not found. It should hold Omnibus.py plus nw.py, hac_var.py, "
            "linchi2.py, cdfchic.py, block_bootstrap.py, FMB_coefficients.py")
    if d not in sys.path:
        sys.path.insert(0, d)
    import Omnibus
    return Omnibus


def omnibus_tests(sentiment_kind="bw", portfolio_kind="25", date_range=COMMON_WINDOW,
                  factor_cols=("s_lag", "mkt_rf", "s_mkt"), verbose=True):
    """Run the scaled CAPM through Kroencke & Thimme's omnibus() toolkit.

    Source: Kroencke, Tim A. & Thimme, Julian (2021), "A Skeptical Appraisal
    of Robust Asset Pricing Tests". Their header asks that the paper be cited
    whenever the code is used -- so cite it.

    WHY THIS EXISTS. Everything else in this module reports an R^2 with no
    standard error, and t-statistics that assume the model is CORRECTLY
    SPECIFIED. Both assumptions are doing a lot of work:

      5.14  Kan/Robotti/Shanken standard error of the sample cross-sectional
            R^2. Without it, comparing "BW gets 0.82" against "MCSI gets
            0.54" is comparing two point estimates with no idea whether they
            differ. KRS (2013, JF) showed this R^2 is estimated very
            imprecisely.
      5.13  KRS test of H0: R^2 = 0.
      3.07  KRS MISSPECIFICATION-ROBUST standard errors. Ordinary
            Fama-MacBeth and Shanken SEs are only valid if the model is
            true. Given the GLS R^2, the chi^2 rejections and the sign flip
            in lambda_s_mkt under reweighting, that is not a safe assumption
            here -- so these are the more honest t-statistics.
      3.03  Shanken SEs, computed by their code. Included purely as a
            cross-check that our panel is being handed over correctly: this
            should reproduce run()'s own Shanken column.

    NOTE on their options: omnibus() hardcodes lags=3 (matching our HAC
    choice) and traded_f=1 internally. traded_f only selects a default null
    value for Lambda0, and since we leave Lambda0 at its default of 0 the
    setting never binds -- every test below is against H0: parameter = 0."""
    Omni = _load_omnibus()
    panel, port_cols = build_panel(sentiment_kind, portfolio_kind, date_range)
    R = panel[port_cols].values                 # T x N excess returns
    f = panel[list(factor_cols)].values         # T x K factors
    T, N, K = len(panel), len(port_cols), len(factor_cols)

    out = {}
    for m in (3.03, 3.07, 5.13, 5.14):
        try:
            out[m] = Omni.omnibus(R, f, m)
        except Exception as exc:                # noqa: BLE001 -- report, don't abort
            out[m] = {"error": f"{type(exc).__name__}: {exc}"}

    def _theta(a):
        """omnibus returns the intercept as 'const' and the K factor premia as
        'Lambda' SEPARATELY, while its SE/T vectors are length K+1 with the
        intercept first. Stitch them back together so labels line up."""
        return np.concatenate([np.ravel(a["const"]), np.ravel(a["Lambda"])])

    if verbose:
        print(f"\n{'='*78}\nOMNIBUS TESTS (Kroencke & Thimme)  --  sentiment = "
              f"{sentiment_kind.upper()}  ({portfolio_kind} portfolios)\n{'='*78}")
        print(f"Sample: {panel.index.min()} - {panel.index.max()}  "
              f"(T = {T}, N = {N}, K = {K})\n")

        a14 = out[5.14]
        if "error" not in a14:
            # NOTE the pairing: omnibus reports R2 (no intercept) and R2i (WITH
            # intercept). Method 5.14's formula uses R2i, and R2i is what matches
            # run()'s own r2 -- so the SE belongs to R2i, not to R2.
            r2i = float(np.ravel(a14["R2i"])[0])
            sev = float(np.ravel(a14["SE"])[0])
            lo, hi = r2i - 1.96 * sev, r2i + 1.96 * sev
            print("-- Cross-sectional R^2 with a STANDARD ERROR (KRS, method 5.14) --")
            print(f"   R^2 (with intercept) = {r2i:.4f}   SE = {sev:.4f}")
            print(f"   95% CI = [{lo:.4f}, {min(hi, 1.0):.4f}]"
                  f"{'  (upper bound truncated at 1)' if hi > 1 else ''}")
            print(f"   for reference: R^2 without intercept = "
                  f"{float(np.ravel(a14['R2'])[0]):.4f}, "
                  f"GLS R^2 = {float(np.ravel(a14['R2i_gls'])[0]):.4f}")
            print("   The interval is the point. Comparing bare R^2 across models"
                  " says little\n   when each one carries an interval this wide.\n")

        if "error" in out[5.13]:
            print(f"-- H0: R^2 = 0 (method 5.13): unavailable -- {out[5.13]['error']}")
            print("   (their own header flags known numerical issues in some"
                  " methods; 5.14 above\n   still gives the SE, which is the"
                  " part that matters here.)\n")

        names = ["const"] + list(factor_cols)
        for m, lbl in ((3.03, "Shanken SEs, THEIR implementation (cross-check)"),
                       (3.07, "KRS misspecification-ROBUST SEs")):
            a = out[m]
            if "error" in a:
                print(f"-- method {m}: {a['error']}\n")
                continue
            th, se_, tt = _theta(a), np.ravel(a["SE"]), np.ravel(a["T"])
            print(f"-- {lbl} (method {m}) --")
            for i, nm in enumerate(names):
                print(f"   {nm:8s} est={th[i]: .4f}  SE={se_[i]:.4f}  t={tt[i]: .3f}")
            print()

        print("-- Reading the two SE columns together --")
        print("   3.03 assumes the model is CORRECTLY specified; 3.07 does not.")
        print("   Given the GLS R^2, the chi2 rejections and the sign flip in")
        print("   lambda_s_mkt under reweighting, 3.07 is the more defensible one.")
        print("   NOTE: their 3.03 applies the Shanken multiplier to the INTERCEPT")
        print("   as well, which run() deliberately does not -- so the constant's")
        print("   t differs between the two by construction. The factor lambdas and")
        print("   both R^2 measures match run() exactly.\n")

    return {"panel": panel, "port_cols": port_cols, "results": out}

In [ ]:
# error bars on the R^2, plus misspecification-robust standard errors
_ = omnibus_tests("bw", portfolio_kind="25")

---
## 16c. The zero-intercept restriction — arguably the sharpest diagnostic here

There is a genuine debate about whether the cross-sectional regression should
include a constant, and it matters more for these results than anything else
we measure.

**The case against a constant.** The left-hand side is an *excess* return, so a
true factor model says

$$E[R^e_i] = \beta_i' \lambda \qquad \text{with no free constant}$$

An asset with zero betas must earn zero excess return — that is what "excess"
means. So $\lambda_0 = 0$ is one of the model's **testable predictions**, not a
technicality, and letting the intercept float relaxes the test.

**The case for one.** Black's (1972) zero-beta CAPM: if investors cannot borrow
at the riskless rate, the zero-beta rate differs from the T-bill rate, which
makes a free intercept theoretically correct rather than a fudge. And under
misspecification, forcing $\lambda_0 = 0$ dumps all the level error onto the
factor lambdas. Which is why most papers — Doukas & Han included — report the
with-intercept version.

`intercept_restriction()` reports both, so the gap is visible rather than
buried in a convention.

**What it shows.** On the 25 portfolios, four of five indices carry constants
of 0.99–1.36 with $t$ from 2.2 to 4.8, and collapse to $R^2$ between −31 and
−59 once the intercept is removed. That constant was absorbing the entire
*level* of average excess returns, leaving the factors to explain only the
spread around it. BW is the exception — constant 0.086, $t = 0.28$ — and still
manages $R^2 = 0.59$ unrestricted, because its factors were pricing the level
too. On the 48 industries, **no index satisfies the restriction at all**.

**And this is in the paper's own Table 3.** Their constants: 1.54 (CAPM), 1.71
(FF3), 1.04 (MSCI), 1.68 (CBCCI), 1.24 (AS) — all strongly significant —
against BW at 0.33 with a Shanken $t$ of 0.72, the only insignificant one in
the table. Our MCSI constant of 1.077 sits right next to their 1.04. The
pattern is theirs as much as ours; they simply do not draw attention to it.

This gives a cleaner statement of the whole project than any $R^2$ comparison:
across five sentiment indices and several test-asset sets, **exactly one
specification satisfies the model's own zero-intercept restriction — BW on the
25 portfolios.** Everything else needs a large free constant to look
respectable, which is another way of saying its factors do not price the level
of returns at all.

In [ ]:
_OMNIBUS_BASE_KEYS = {"beta", "alpha", "Lambda", "Lambda_gls", "const", "const_gls",
                      "PE", "PEi", "PE_gls", "PEi_gls", "R2", "R2i", "R2_gls",
                      "R2i_gls", "AV"}


def omnibus_method(method, sentiment_kind="bw", portfolio_kind="25",
                   date_range=COMMON_WINDOW, factor_cols=("s_lag", "mkt_rf", "s_mkt"),
                   verbose=True):
    """Run ONE omnibus() method on this project's panel, by number.

        omnibus_method(1.01)      # betas all zero? (iid residuals)
        omnibus_method(3.07)      # KRS misspecification-robust SEs
        omnibus_method(5.14)      # standard error of the cross-sectional R^2

    THE TRAP THIS GUARDS AGAINST. omnibus() dispatches on an exact FLOAT
    comparison (`if method == 1.01:`). So 1.1 is NOT 1.01, and 1.2 is NOT
    1.02 -- and when nothing matches, their code does not raise. It simply
    returns the base result dict with none of the method-specific keys, so a
    typo looks like a successful run that inexplicably has no test statistic.
    This wrapper raises instead.

    Method numbers, from Omnibus.py's own header:
      1.01-1.06  preliminary: are the betas all zero / all equal?
      2.01-2.10  cross-sectional regression WITHOUT intercept
      3.01-3.10  cross-sectional regression WITH intercept
                 (3.06 Giglio/Xiu three-pass, 3.07 Kan/Robotti/Shanken robust)
      4.01-4.21  pricing-error tests without intercept (GRS, chi2, FAR, ...)
      5.05-5.18  pricing-error tests with intercept
                 (5.11-5.14 the KRS R^2 tests, 5.14 its standard error)
      6.01-6.04  SDF loadings (what sdf_gmm.py reimplements)

    Known to fail inside their code on this data: 5.13. Their header also
    flags 2.09/3.09 and 6.03/6.04."""
    Omni = _load_omnibus()
    panel, port_cols = build_panel(sentiment_kind, portfolio_kind, date_range)
    R = panel[port_cols].values
    f = panel[list(factor_cols)].values

    ans = Omni.omnibus(R, f, float(method))
    extra = [k for k in ans.keys() if k not in _OMNIBUS_BASE_KEYS]
    if not extra:
        raise ValueError(
            f"omnibus() returned no results for method={method!r}. Their dispatch "
            f"is an exact float match, so the number must be written with TWO "
            f"decimals -- 1.01 not 1.1, 3.07 not 3.7. Nothing matched {float(method)}.")

    if verbose:
        print(f"\n{'='*72}\nomnibus method {method}  --  {sentiment_kind.upper()}, "
              f"{portfolio_kind} portfolios\n{'='*72}")
        print(f"T = {len(panel)}, N = {len(port_cols)}, K = {len(factor_cols)}")
        print(f"method-specific keys: {extra}\n")
        names = ["const"] + list(factor_cols)
        for k in extra:
            v = np.ravel(ans[k])
            if v.size == len(names):          # one value per coefficient
                for nm, x in zip(names, v):
                    print(f"   {k:6s} {nm:8s} {x: .4f}")
            elif v.size == len(factor_cols):  # factors only, no intercept
                for nm, x in zip(factor_cols, v):
                    print(f"   {k:6s} {nm:8s} {x: .4f}")
            elif v.size == 1:
                print(f"   {k:6s} {float(v[0]): .4f}")
            else:
                print(f"   {k:6s} (len {v.size}) {np.round(v, 4)}")
    return ans

In [ ]:
# Run any single omnibus method by number. Two decimals, always: 1.01 not 1.1.
_ = omnibus_method(1.01)     # preliminary: are the betas all zero?
# _ = omnibus_method(3.07)   # KRS misspecification-robust standard errors
# _ = omnibus_method(5.14)   # standard error of the cross-sectional R^2

In [ ]:
def intercept_restriction(kinds=("bw", "mcsi", "cbcci", "pmi", "cfnai"),
                          portfolio_kind="25", date_range=COMMON_WINDOW,
                          factor_cols=("s_lag", "mkt_rf", "s_mkt"), verbose=True):
    """Test the model's OWN zero-intercept restriction, index by index.

    THE POINT. The left-hand side here is an EXCESS return, so a true factor
    model says

        E[R^e_i] = beta_i' lambda        -- with NO free constant

    An asset with zero betas must earn zero excess return; that is what
    "excess" means, and it is one of the model's testable predictions rather
    than a technicality. Estimating with a free intercept relaxes that test.

    The counter-argument is real: Black's (1972) zero-beta CAPM says that
    without riskless borrowing the zero-beta rate differs from the T-bill
    rate, which makes a free intercept correct rather than a fudge. And under
    misspecification, forcing lambda_0 = 0 pushes all the level error onto the
    factor lambdas. Which is why most papers -- Doukas & Han included --
    report the with-intercept version. This function reports BOTH so the gap
    is visible instead of buried in a convention.

    HOW TO READ IT. A large, significant constant means the intercept is
    absorbing the LEVEL of average excess returns, leaving the factors to
    explain only the spread around it. When that is happening, dropping the
    intercept does not weaken the model slightly -- it collapses it.

    On the 25 portfolios over COMMON_WINDOW, four of five indices carry
    constants of 0.99-1.36 (t between 2.2 and 4.8) and fall to R^2 of -31 to
    -59 once the intercept goes. BW is the exception: constant 0.086 (t=0.28),
    and it still posts R^2 = 0.59 unrestricted, because its factors were
    already pricing the level and not just the dispersion.

    This pattern is in the PAPER's own Table 3 too -- their constants are
    1.54 (CAPM), 1.71 (FF3), 1.04 (MSCI), 1.68 (CBCCI), 1.24 (AS), all
    strongly significant, against BW at 0.33 with a Shanken t of 0.72, the
    only insignificant one in the table. They do not draw attention to it."""
    Omni = _load_omnibus()
    rows = []
    for k in kinds:
        panel, port_cols = build_panel(k, portfolio_kind, date_range)
        R = panel[port_cols].values
        f = panel[list(factor_cols)].values
        a = Omni.omnibus(R, f, 3.03)

        # recompute the constant and its t the same way run() does, inline --
        # calling run() here would print a full report per index. NOTE we use
        # OUR convention, where the Shanken multiplier is applied to the factor
        # lambdas but NOT to the intercept; omnibus applies it to the intercept
        # too, so its t(const) is smaller. See omnibus_tests().
        beta_mat, _ = first_stage_betas(panel, port_cols, factor_cols)
        _, lam_mean, se_fm, _ = fama_macbeth(panel, port_cols, beta_mat)
        rows.append({
            "sentiment": k.upper(),
            "const": lam_mean["const"],
            "t_const": lam_mean["const"] / se_fm["const"],
            "R2_with_int": float(np.ravel(a["R2i"])[0]),
            "R2_no_int": float(np.ravel(a["R2"])[0]),
            "GLS_with_int": float(np.ravel(a["R2i_gls"])[0]),
            "GLS_no_int": float(np.ravel(a["R2_gls"])[0]),
            "T": len(panel),
        })
    tbl = pd.DataFrame(rows).set_index("sentiment")

    if verbose:
        print(f"\n{'='*84}\nZERO-INTERCEPT RESTRICTION  --  {portfolio_kind} portfolios"
              f"\n{'='*84}")
        print("With excess returns on the LHS the model predicts const = 0.")
        print("A big significant const means the intercept, not the factors,"
              " is pricing the level.\n")
        print(tbl.round(4).to_string())
        ok = tbl[tbl["t_const"].abs() < 1.96]
        print(f"\nSatisfies the restriction (|t(const)| < 1.96): "
              f"{', '.join(ok.index) if len(ok) else 'NONE'}")
    return tbl

In [ ]:
_ = intercept_restriction()                              # 25 size-BM portfolios
# _ = intercept_restriction(portfolio_kind="industry48") # none pass here

In [ ]:
def run_all_sentiments(portfolio_kind="industry48", threshold="1sd", date_range=COMMON_WINDOW):
    """Convenience driver: runs the sentiment-scaled CAPM (Eq.9 spec, via
    run()) individually for the 5 sentiment indices -- MCSI, CBCCI, PMI, BW,
    CFNAI -- all on the SAME test-asset set and the SAME common date window
    (default COMMON_WINDOW = Dec 1969-Dec 2025), so the 5 results are
    directly comparable. Deliberately excludes the composite AS index this
    round (it's built FROM BW/MCSI/CBCCI, so including it alongside those
    three would be double-counting the same information)."""
    kinds = ["mcsi", "cbcci", "pmi", "bw", "cfnai"]
    results = {}
    rows = []
    for kind in kinds:
        res = run(kind, threshold=threshold, portfolio_kind=portfolio_kind, date_range=date_range)
        results[kind] = res
        lam = res["lambda_table"]
        rows.append({
            "sentiment": kind.upper(),
            "lam_s_lag": lam.loc["s_lag", "lambda"], "t_s_lag": lam.loc["s_lag", "t_Shanken"],
            "lam_mkt_rf": lam.loc["mkt_rf", "lambda"], "t_mkt_rf": lam.loc["mkt_rf", "t_Shanken"],
            "lam_s_mkt": lam.loc["s_mkt", "lambda"], "t_s_mkt": lam.loc["s_mkt", "t_Shanken"],
            "R2": res["r2"], "R2_gls": res["r2_gls"],
            "chi2_p": res["chi2"][2],
        })
    summary = pd.DataFrame(rows).set_index("sentiment")
    print(f"\n{'='*78}\nSUMMARY -- 5 sentiment-scaled CAPMs, common window, {portfolio_kind} portfolios\n{'='*78}")
    print(summary.round(4))
    return {"results": results, "summary": summary}

---
## 17. Reproducing each table

Every call below is independent — run whichever you need.

In [ ]:
# ---- Table 3 analogue: scaled CAPM, five indices, 25 size-BM portfolios
out_25 = run_all_sentiments(portfolio_kind="25")

# ---- the same on 48 industry portfolios (the hard test)
# out_ind = run_all_sentiments(portfolio_kind="industry48")

# ---- unscaled FF3 baseline, both asset sets
# run_ff3_plain("mcsi", portfolio_kind="25",         date_range=COMMON_WINDOW)
# run_ff3_plain("mcsi", portfolio_kind="industry48", date_range=COMMON_WINDOW)

# ---- Table 11 analogue: sentiment-scaled FF3
# run_ff3("bw", portfolio_kind="25", date_range=COMMON_WINDOW)

# ---- Table 2 analogue: predictive regressions
# run_table2(date_range=COMMON_WINDOW)

# ---- Table 13 analogue: PLS robustness
# run_table13("pls", portfolio_kind="25", date_range=COMMON_WINDOW)

# ---- Tables 8/9 analogue: anomaly portfolios (partial -- see section 14)
# run_anomaly_tables("as")

# ---- sector cuts (see the aggregation warning in section 1 before trusting the fit)
# run("bw", portfolio_kind="industry12", date_range=COMMON_WINDOW)
# run_all_sentiments(portfolio_kind="industry12")

# ---- AAII, the direct investor-sentiment survey (shorter sample: 1987:07 on)
# run("aaii", portfolio_kind="25", date_range=COMMON_WINDOW)

# ---- extended macro controls (EXTENSIONS beyond the paper -- see section 15)
# run_table2(date_range=COMMON_WINDOW, controls="full")   # 12 independent predictors
# run_table2(date_range=COMMON_WINDOW, controls="pca", n_pc=3)
# run_table2(date_range=COMMON_WINDOW, add_cay=True)      # short sample (~597 months)
# run_table2(date_range=COMMON_WINDOW, add_macro=True)    # unemployment + housing + CAPE yield

# ---- does sentiment survive controlling for the real economy? (see section 15)
# predictive_regression("bw", date_range=COMMON_WINDOW, add_macro=True)

---
## 18. Reference: paper → code

| Paper | Where | This notebook |
|---|---|---|
| Eq. 9, scaled CAPM pricing equation | p. 215 | `run()` (§11) |
| Table 3, FM regressions on 25 portfolios | p. 221 | `run()`, `run_ff3_plain()` |
| Tables 5–7, state-beta SML | — | `state_beta_analysis()` (§10) |
| Tables 8/9, anomaly portfolios | — | `run_anomaly_tables()` (§14, partial) |
| Table 11, sentiment-scaled FF3 | — | `run_ff3()` (§12) |
| Table 2, predictive regression (Eq. 11) | — | `run_table2()` (§15) |
| Table 13, PLS robustness | — | `run_table13()` (§15) |
| Shanken correction, footnote 16 | — | `shanken_correction()` (§8) |
| $\chi^2$ test, footnote 18 | — | `chi2_joint_test()` (§9.3) |
| Controls, footnote 21 | — | `load_goyal_controls()` (§3.4) |
| CAY (Lettau-Ludvigson) | footnote 21 | `load_cay()` (§3.4), opt-in via `add_cay=True` |
| *(not in the paper)* | — | `load_aaii()`, `load_goyal_full()`, `load_macro()` — extensions |

## Known deviations from the paper

1. **Sample** — 1969:12–2025:12 here versus 1965:07–2015:09 in the paper.
2. **Market return** — Ken French `Mkt-RF`; the paper uses the CRSP value-weighted
   excess return. Close but not identical.
3. **CBCCI source** — investing.com export (repaired; see §4), not the paper's source.
4. **CAY control omitted** — not present in Goyal's dataset.
5. **Term premium** — `lty - tbl` rather than 20yr minus 1yr.
6. **Anomalies** — 4 public proxies rather than the paper's 8 WRDS-built measures.
7. **AS loadings** — re-estimated by PCA on this sample rather than the paper's
   published 0.318 / 0.443 / 0.452.
8. **CAY** — available but not a default control; the series ends 2019Q3, so
   including it costs ~75 months.

Beyond the replication, three **extensions** are wired in and clearly marked
as such: the AAII investor-sentiment survey (§3.4), the extended Welch-Goyal
control sets via `controls="full"` / `"pca"`, and the real-activity/valuation
block via `add_macro=True` (both §15). None is used by default, so every
headline number remains a faithful reproduction.

The macro extension produced the most consequential finding here: sentiment's
predictive coefficient collapses once the CAPE earnings yield is controlled
for, which suggests a good deal of what looks like a sentiment effect may be
a valuation effect wearing a different label.

Given all of that, the replication tracks the paper closely on the 25 portfolios:
signs match throughout, CBCCI's $\lambda_m$ comes out at −0.746 against their
−0.74, MCSI's intercept at 1.08 against their 1.04, and the unadjusted and
adjusted $R^2$ line up across all four models.